In [ ]:
#STEP 1: Environment setup + dataset loading

import os
import sys
import json
import time
import random
import hashlib
import warnings
import platform
import datetime
from pathlib import Path
import numpy as np
import pandas as pd

#Optional display settings for cleaner notebook outputs
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

warnings.filterwarnings("ignore")

#Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("=" * 80)
print("STEP 1: ENVIRONMENT + DATASET INSPECTION")
print("=" * 80)
print("\n[Environment]")
print("Python version      :", sys.version)
print("Platform            :", platform.platform())
print("NumPy version       :", np.__version__)
print("Pandas version      :", pd.__version__)
print("Random seed         :", RANDOM_SEED)

#input: dataset path
DATA_PATH = "../datasets/ton-iot-train_test_network.csv"  

print("\n[Dataset path]")
print(DATA_PATH)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset file not found: {DATA_PATH}")

file_size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
print(f"File size           : {file_size_mb:.2f} MB")

#Safe dataset loading
t0 = time.perf_counter()

try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
except UnicodeDecodeError:
    print("UTF-8 loading failed. Retrying with latin1 encoding...")
    df = pd.read_csv(DATA_PATH, low_memory=False, encoding="latin1")
except Exception as e:
    raise RuntimeError(f"Failed to load dataset: {e}")

load_time = time.perf_counter() - t0

print("\n[Load status]")
print(f"Dataset loaded successfully in {load_time:.4f} seconds")

#Basic shape and preview
print("\n[Basic dataset summary]")
print("Rows                :", df.shape[0])
print("Columns             :", df.shape[1])

print("\n[Column names]")
print(df.columns.tolist())

print("\n[First 5 rows]")
print(df.head())

print("\n[Last 5 rows]")
print(df.tail())

print("\n[Random 3 rows]")
print(df.sample(min(3, len(df)), random_state=RANDOM_SEED))

#Column-level metadata
print("\n[Column data types]")
dtype_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_percent": (100 * df.isna().mean()).round(4).values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(by=["dtype", "null_percent", "n_unique"], ascending=[True, False, False])

print(dtype_df)

#Memory usage
mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print("\n[Memory usage]")
print(f"Approximate in-memory size: {mem_mb:.2f} MB")

#Missing values
print("\n[Missing values summary]")
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

if len(missing_counts) == 0:
    print("No missing values detected.")
else:
    missing_df = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": (100 * missing_counts / len(df)).round(4)
    })
    print(missing_df)

#Duplicate analysis
print("\n[Duplicate analysis]")
n_dup = df.duplicated().sum()
dup_pct = 100 * n_dup / len(df) if len(df) > 0 else 0
print("Duplicate rows      :", n_dup)
print(f"Duplicate percent   : {dup_pct:.4f}%")

#Numeric vs non-numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("\n[Column type groups]")
print("Numeric columns     :", len(numeric_cols))
print("Non-numeric columns :", len(non_numeric_cols))

print("\nNumeric columns list:")
print(numeric_cols)

print("\nNon-numeric columns list:")
print(non_numeric_cols)

#Candidate label column detection
print("\n[Candidate label columns]")
candidate_keywords = ["label", "attack", "class", "category", "target", "type"]
candidate_label_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in candidate_keywords)
]
print(candidate_label_cols if candidate_label_cols else "No obvious label column found automatically.")

#For each candidate label column, print class distribution
for col in candidate_label_cols:
    print("\n" + "-" * 80)
    print(f"Candidate label column: {col}")
    print("dtype               :", df[col].dtype)
    print("unique classes      :", df[col].nunique(dropna=False))
    print("top class counts:")
    print(df[col].value_counts(dropna=False).head(20))
    print("class percentages (%):")
    print((100 * df[col].value_counts(dropna=False, normalize=True)).round(4).head(20))

#Low-cardinality non-numeric columns
#Useful for spotting label-like or protocol-like fields
print("\n[Low-cardinality non-numeric columns (<= 25 unique values)]")
low_card_non_num = []
for c in non_numeric_cols:
    nunq = df[c].nunique(dropna=False)
    if nunq <= 25:
        low_card_non_num.append(c)

if len(low_card_non_num) == 0:
    print("None found.")
else:
    for c in low_card_non_num:
        print("\nColumn:", c)
        print("Unique values:", df[c].nunique(dropna=False))
        print(df[c].value_counts(dropna=False).head(25))

#Potential identifier/high-cardinality columns
#Useful for leakage inspection
print("\n[Potential high-cardinality / identifier-like columns]")
high_card_df = pd.DataFrame({
    "column": df.columns,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    "unique_ratio": [df[c].nunique(dropna=True) / len(df) if len(df) > 0 else 0 for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns]
}).sort_values(by="unique_ratio", ascending=False)

print(high_card_df.head(30))

#Basic numeric descriptive stats
if len(numeric_cols) > 0:
    print("\n[Numeric descriptive statistics]")
    print(df[numeric_cols].describe().T)
else:
    print("\n[Numeric descriptive statistics]")
    print("No numeric columns found.")

#Infinite value check in numeric columns
print("\n[Infinite value check]")
if len(numeric_cols) > 0:
    inf_counts = np.isinf(df[numeric_cols].select_dtypes(include=[np.number])).sum()
    inf_counts = inf_counts[inf_counts > 0]
    if len(inf_counts) == 0:
        print("No +inf or -inf values detected in numeric columns.")
    else:
        print(inf_counts.sort_values(ascending=False))
else:
    print("Skipped because no numeric columns were detected.")

#Object/string columns: sample unique values
#Helps identify protocol/attack fields and dirty formatting
print("\n[Object/string columns: example values]")
obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

if len(obj_cols) == 0:
    print("No object/string columns found.")
else:
    for c in obj_cols[:20]:   # keep output manageable
        vals = df[c].dropna().astype(str).unique()[:10]
        print(f"\nColumn: {c}")
        print("Example values:", vals)


# Suspicious columns that may cause leakage
print("\n[Potential leakage-related columns by name pattern]")
leakage_keywords = [
    "label", "attack", "class", "category", "target",
    "src", "dst", "ip", "addr", "port", "flow_id", "session",
    "timestamp", "time", "date", "ts", "uid", "id"
]

suspicious_cols = [c for c in df.columns if any(k in c.lower() for k in leakage_keywords)]
print(suspicious_cols if suspicious_cols else "No suspicious columns found by keyword rule.")

# dataset summary block
print("\n" + "=" * 80)
print("[DATASET SNAPSHOT]")
print("=" * 80)

summary = {
    "dataset_path": DATA_PATH,
    "loaded_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "memory_mb": round(mem_mb, 4),
    "load_time_seconds": round(load_time, 4),
    "duplicate_rows": int(n_dup),
    "duplicate_percent": round(dup_pct, 4),
    "numeric_columns": len(numeric_cols),
    "non_numeric_columns": len(non_numeric_cols),
    "candidate_label_columns": candidate_label_cols,
    "columns_with_missing_values": int((df.isna().sum() > 0).sum())
}

print(json.dumps(summary, indent=2))


# Save quick profiling artifacts
OUT_DIR = "dataset_inspection_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

dtype_df.to_csv(os.path.join(OUT_DIR, "column_profile.csv"), index=False)
high_card_df.to_csv(os.path.join(OUT_DIR, "high_cardinality_columns.csv"), index=False)

if len(missing_counts) > 0:
    missing_df.to_csv(os.path.join(OUT_DIR, "missing_value_summary.csv"))

with open(os.path.join(OUT_DIR, "dataset_snapshot_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n[Artifacts saved]")
print(f"Saved profiling files to: {OUT_DIR}")

print("\nSTEP 1 completed successfully.")


In [ ]:
# STEP 2: Multiclass target preparation + data cleaning +
# leakage-aware feature design + stratified train/val/test split

import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("=" * 80)
print("STEP 2: MULTICLASS TARGET PREPARATION + CLEANING + SPLITTING")
print("=" * 80)

t0 = time.perf_counter()

# Safety checks
assert "df" in globals(), "DataFrame 'df' not found. Run Step 1 first."
assert "type" in df.columns, "Expected multiclass target column 'type' not found."
assert "label" in df.columns, "Expected binary helper column 'label' not found."

df_work = df.copy()

print("\n[Initial dataset shape]")
print(df_work.shape)


# Standardize target text
df_work["type"] = df_work["type"].astype(str).str.strip().str.lower()
df_work["label"] = pd.to_numeric(df_work["label"], errors="coerce").fillna(-1).astype(int)

print("\n[Original multiclass distribution: 'type']")
print(df_work["type"].value_counts(dropna=False))

print("\n[Original binary distribution: 'label']")
print(df_work["label"].value_counts(dropna=False))


# Binary-vs-multiclass consistency check
# normal should correspond to label=0, attacks to label=1
print("\n[Consistency check between 'type' and binary 'label']")
consistency_table = pd.crosstab(df_work["type"], df_work["label"], margins=True)
print(consistency_table)


# Remove exact duplicate rows, because duplicates can inflate performance
n_before = len(df_work)
dup_count = df_work.duplicated().sum()

df_work = df_work.drop_duplicates().reset_index(drop=True)

n_after = len(df_work)
removed = n_before - n_after

print("\n[Duplicate handling]")
print("Rows before deduplication :", n_before)
print("Duplicate rows removed    :", removed)
print("Rows after deduplication  :", n_after)
print(f"Percent removed           : {100 * removed / max(n_before, 1):.4f}%")

print("\n[Class distribution after deduplication]")
print(df_work["type"].value_counts())


# Rare class handling
MIN_CLASS_COUNT = 2000

class_counts = df_work["type"].value_counts()
rare_classes = class_counts[class_counts < MIN_CLASS_COUNT].index.tolist()

print("\n[Rare class analysis]")
print("Minimum class count threshold:", MIN_CLASS_COUNT)
print("Rare classes detected        :", rare_classes if rare_classes else "None")

df_main = df_work[~df_work["type"].isin(rare_classes)].copy().reset_index(drop=True)

print("\n[Dataset shape after rare-class filtering]")
print(df_main.shape)

print("\n[Retained multiclass distribution]")
print(df_main["type"].value_counts())

excluded_summary = {
    "rare_classes_removed": rare_classes,
    "rows_removed_due_to_rare_classes": int(len(df_work) - len(df_main))
}
print("\n[Excluded class summary]")
print(json.dumps(excluded_summary, indent=2))


# Preserve context columns for forensic graph/evidence pipeline
# These are NOT all necessarily model features.
CONTEXT_COLS = [c for c in [
    "src_ip", "dst_ip", "src_port", "dst_port", "proto", "service", "conn_state"
] if c in df_main.columns]

print("\n[Context columns preserved for forensic use]")
print(CONTEXT_COLS)


# Define target and feature exclusions
# - 'type' is multiclass target
# - 'label' is binary helper and must be excluded from features
# - raw IPs are excluded from model features to reduce leakage / memorization
TARGET_COL = "type"
BINARY_HELPER_COL = "label"

DROP_FROM_FEATURES = [c for c in [
    TARGET_COL,
    BINARY_HELPER_COL,
    "src_ip",
    "dst_ip"
] if c in df_main.columns]

print("\n[Columns dropped from model features]")
print(DROP_FROM_FEATURES)


# Build feature table
# Replace '-' placeholders in object columns to make categorical handling explicit
X_raw = df_main.drop(columns=DROP_FROM_FEATURES, errors="ignore").copy()
y_text = df_main[TARGET_COL].copy()

# normalize placeholder tokens in object columns
obj_cols = X_raw.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols:
    X_raw[c] = X_raw[c].astype(str).str.strip()
    X_raw[c] = X_raw[c].replace({"-": "MISSING_TOKEN", "": "MISSING_TOKEN", "nan": "MISSING_TOKEN"})

print("\n[Raw feature matrix]")
print("Shape:", X_raw.shape)

print("\n[Feature dtypes before encoding]")
print(X_raw.dtypes.value_counts())


# Encode multiclass target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

label_mapping = {cls: int(idx) for idx, cls in enumerate(label_encoder.classes_)}
inv_label_mapping = {int(idx): cls for cls, idx in label_mapping.items()}

print("\n[Multiclass label mapping]")
print(label_mapping)

print("\n[Encoded target distribution]")
print(pd.Series(y).value_counts().sort_index())


# Column groups for modeling
numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()

print("\n[Model feature groups]")
print("Numeric feature count     :", len(numeric_cols))
print("Categorical feature count :", len(categorical_cols))

print("\nNumeric feature columns:")
print(numeric_cols)

print("\nCategorical feature columns:")
print(categorical_cols)


# Cardinality review of categorical columns
# Useful for deciding one-hot vs frequency encoding later
cat_cardinality = pd.DataFrame({
    "column": categorical_cols,
    "n_unique": [X_raw[c].nunique(dropna=False) for c in categorical_cols]
}).sort_values(by="n_unique", ascending=False)

print("\n[Categorical cardinality summary]")
print(cat_cardinality)


# Optional: identify very high-cardinality categorical columns
# We may frequency-encode these later instead of one-hot encoding
HIGH_CARD_THRESHOLD = 50
high_card_cat_cols = cat_cardinality.loc[
    cat_cardinality["n_unique"] > HIGH_CARD_THRESHOLD, "column"
].tolist()

low_mid_card_cat_cols = cat_cardinality.loc[
    cat_cardinality["n_unique"] <= HIGH_CARD_THRESHOLD, "column"
].tolist()

print("\n[Categorical encoding strategy suggestion]")
print("High-cardinality categorical columns  :", high_card_cat_cols if high_card_cat_cols else "None")
print("Low/mid-cardinality categorical cols  :", low_mid_card_cat_cols if low_mid_card_cat_cols else "None")


# Stratified split: train / val / test
# First stable setup for multiclass pipeline
X_trainval_raw, X_test_raw, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
    X_raw,
    y,
    df_main.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_raw, X_val_raw, y_train, y_val, idx_train, idx_val = train_test_split(
    X_trainval_raw,
    y_trainval,
    idx_trainval,
    test_size=0.25,   # 0.25 of 0.80 = 0.20 overall
    random_state=42,
    stratify=y_trainval
)

# context frames aligned with split indices
ctx_train = df_main.loc[idx_train, CONTEXT_COLS].reset_index(drop=True)
ctx_val   = df_main.loc[idx_val, CONTEXT_COLS].reset_index(drop=True)
ctx_test  = df_main.loc[idx_test, CONTEXT_COLS].reset_index(drop=True)

# reset feature indices for cleaner downstream use
X_train_raw = X_train_raw.reset_index(drop=True)
X_val_raw   = X_val_raw.reset_index(drop=True)
X_test_raw  = X_test_raw.reset_index(drop=True)

y_train = pd.Series(y_train).reset_index(drop=True)
y_val   = pd.Series(y_val).reset_index(drop=True)
y_test  = pd.Series(y_test).reset_index(drop=True)


# Split diagnostics
print("\n[Split shapes]")
print("X_train_raw :", X_train_raw.shape)
print("X_val_raw   :", X_val_raw.shape)
print("X_test_raw  :", X_test_raw.shape)

def print_split_distribution(name, y_split):
    vc = pd.Series(y_split).value_counts().sort_index()
    pct = pd.Series(y_split).value_counts(normalize=True).sort_index() * 100
    df_dist = pd.DataFrame({
        "class_id": vc.index,
        "class_name": [inv_label_mapping[i] for i in vc.index],
        "count": vc.values,
        "percent": pct.values.round(4)
    })
    print(f"\n[{name} class distribution]")
    print(df_dist)

print_split_distribution("TRAIN", y_train)
print_split_distribution("VAL", y_val)
print_split_distribution("TEST", y_test)


# Check for unseen categorical levels between splits
# Important for robust preprocessing
print("\n[Unseen-category diagnostic: validation/test vs training]")
unseen_report = []

for c in categorical_cols:
    train_vals = set(X_train_raw[c].astype(str).unique())
    val_vals = set(X_val_raw[c].astype(str).unique())
    test_vals = set(X_test_raw[c].astype(str).unique())

    unseen_val = sorted(list(val_vals - train_vals))
    unseen_test = sorted(list(test_vals - train_vals))

    unseen_report.append({
        "column": c,
        "n_train_unique": len(train_vals),
        "n_val_unseen": len(unseen_val),
        "n_test_unseen": len(unseen_test)
    })

unseen_report_df = pd.DataFrame(unseen_report).sort_values(
    by=["n_test_unseen", "n_val_unseen", "n_train_unique"],
    ascending=[False, False, False]
)

print(unseen_report_df)


# Save split and metadata artifacts
OUT_DIR = "multiclass_preparation_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

cat_cardinality.to_csv(os.path.join(OUT_DIR, "categorical_cardinality.csv"), index=False)
unseen_report_df.to_csv(os.path.join(OUT_DIR, "unseen_category_report.csv"), index=False)

prep_summary = {
    "target_column": TARGET_COL,
    "binary_helper_column": BINARY_HELPER_COL,
    "label_mapping": label_mapping,
    "rare_classes_removed": rare_classes,
    "rows_after_deduplication": int(len(df_work)),
    "rows_after_rare_class_filter": int(len(df_main)),
    "n_features_raw": int(X_raw.shape[1]),
    "numeric_feature_count": int(len(numeric_cols)),
    "categorical_feature_count": int(len(categorical_cols)),
    "high_cardinality_categorical_columns": high_card_cat_cols,
    "context_columns": CONTEXT_COLS,
    "train_shape": list(X_train_raw.shape),
    "val_shape": list(X_val_raw.shape),
    "test_shape": list(X_test_raw.shape)
}

with open(os.path.join(OUT_DIR, "multiclass_prep_summary.json"), "w", encoding="utf-8") as f:
    json.dump(prep_summary, f, indent=2)

t_step2 = time.perf_counter() - t0

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Preparation summary]")
print(json.dumps(prep_summary, indent=2))

print(f"\nSTEP 2 completed successfully in {t_step2:.4f} seconds.")

In [ ]:
# STEP 2B: Companion preparation
# - all-classes retained dataset
# - leakage-resistant grouped split metadata


import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

print("=" * 80)
print("STEP 2B: REQUIRED COMPANION PREPARATION")
print("=" * 80)

assert "df_work" in globals(), "Run Step 2 first."
assert "label_encoder" in globals(), "Run Step 2 first."


# A. Retain all classes for robustness run
df_all = df_work.copy().reset_index(drop=True)

TARGET_COL_ALL = "type"
BINARY_HELPER_COL_ALL = "label"

DROP_FROM_FEATURES_ALL = [c for c in [
    TARGET_COL_ALL,
    BINARY_HELPER_COL_ALL,
    "src_ip",
    "dst_ip"
] if c in df_all.columns]

X_all_raw = df_all.drop(columns=DROP_FROM_FEATURES_ALL, errors="ignore").copy()
y_all_text = df_all[TARGET_COL_ALL].copy()

obj_cols_all = X_all_raw.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols_all:
    X_all_raw[c] = X_all_raw[c].astype(str).str.strip()
    X_all_raw[c] = X_all_raw[c].replace({"-": "MISSING_TOKEN", "": "MISSING_TOKEN", "nan": "MISSING_TOKEN"})

# fresh encoder for all-classes setup
from sklearn.preprocessing import LabelEncoder
label_encoder_all = LabelEncoder()
y_all = label_encoder_all.fit_transform(y_all_text)

label_mapping_all = {cls: int(i) for i, cls in enumerate(label_encoder_all.classes_)}
inv_label_mapping_all = {int(i): cls for cls, i in label_mapping_all.items()}

print("\n[All-classes setup]")
print("Shape:", df_all.shape)
print("Class mapping:", label_mapping_all)
print("Class counts:")
print(df_all["type"].value_counts())

# class weights for rare-class robustness run
classes_all = np.unique(y_all)
class_weights_all = compute_class_weight(
    class_weight="balanced",
    classes=classes_all,
    y=y_all
)
class_weight_dict_all = {int(c): float(w) for c, w in zip(classes_all, class_weights_all)}

print("\n[Suggested class weights for all-classes run]")
print(class_weight_dict_all)


# B. Leakage-resistant grouped split
# Group by src_ip if available, else fallback to dst_ip
GROUP_COL = "src_ip" if "src_ip" in df_main.columns else ("dst_ip" if "dst_ip" in df_main.columns else None)
assert GROUP_COL is not None, "No suitable grouping column found for grouped split."

groups = df_main[GROUP_COL].astype(str).values

print("\n[Leakage-resistant grouped split setup]")
print("Grouping column:", GROUP_COL)
print("Rows           :", len(groups))
print("Unique groups  :", len(np.unique(groups)))

# Use group shuffle for train/test, then split train into train/val by groups
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx_g, test_idx_g = next(gss_outer.split(X_raw, y, groups=groups))

X_trainval_group_raw = X_raw.iloc[trainval_idx_g].reset_index(drop=True)
X_test_group_raw     = X_raw.iloc[test_idx_g].reset_index(drop=True)

y_trainval_group = pd.Series(y).iloc[trainval_idx_g].reset_index(drop=True)
y_test_group     = pd.Series(y).iloc[test_idx_g].reset_index(drop=True)

groups_trainval = groups[trainval_idx_g]

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_g2, val_idx_g2 = next(gss_inner.split(X_trainval_group_raw, y_trainval_group, groups=groups_trainval))

X_train_group_raw = X_trainval_group_raw.iloc[train_idx_g2].reset_index(drop=True)
X_val_group_raw   = X_trainval_group_raw.iloc[val_idx_g2].reset_index(drop=True)

y_train_group = y_trainval_group.iloc[train_idx_g2].reset_index(drop=True)
y_val_group   = y_trainval_group.iloc[val_idx_g2].reset_index(drop=True)

ctx_train_group = df_main.iloc[np.array(trainval_idx_g)[train_idx_g2]][CONTEXT_COLS].reset_index(drop=True)
ctx_val_group   = df_main.iloc[np.array(trainval_idx_g)[val_idx_g2]][CONTEXT_COLS].reset_index(drop=True)
ctx_test_group  = df_main.iloc[test_idx_g][CONTEXT_COLS].reset_index(drop=True)

print("\n[Grouped split shapes]")
print("X_train_group_raw :", X_train_group_raw.shape)
print("X_val_group_raw   :", X_val_group_raw.shape)
print("X_test_group_raw  :", X_test_group_raw.shape)

def show_group_dist(name, y_split, inv_map):
    vc = pd.Series(y_split).value_counts().sort_index()
    pct = pd.Series(y_split).value_counts(normalize=True).sort_index() * 100
    dist_df = pd.DataFrame({
        "class_id": vc.index,
        "class_name": [inv_map[int(i)] for i in vc.index],
        "count": vc.values,
        "percent": pct.values.round(4)
    })
    print(f"\n[{name} grouped-split class distribution]")
    print(dist_df)

show_group_dist("TRAIN", y_train_group, inv_label_mapping)
show_group_dist("VAL", y_val_group, inv_label_mapping)
show_group_dist("TEST", y_test_group, inv_label_mapping)

# group overlap diagnostic
train_groups_final = set(ctx_train_group[GROUP_COL].astype(str).unique())
val_groups_final   = set(ctx_val_group[GROUP_COL].astype(str).unique())
test_groups_final  = set(ctx_test_group[GROUP_COL].astype(str).unique())

print("\n[Group overlap check]")
print("Train ∩ Val  :", len(train_groups_final & val_groups_final))
print("Train ∩ Test :", len(train_groups_final & test_groups_final))
print("Val ∩ Test   :", len(val_groups_final & test_groups_final))


# Save metadata
OUT_DIR_2B = "companion_outputs"
os.makedirs(OUT_DIR_2B, exist_ok=True)

companion_summary = {
    "all_classes_shape": list(df_all.shape),
    "all_classes_label_mapping": label_mapping_all,
    "all_classes_class_weights": class_weight_dict_all,
    "group_column": GROUP_COL,
    "n_unique_groups": int(len(np.unique(groups))),
    "grouped_train_shape": list(X_train_group_raw.shape),
    "grouped_val_shape": list(X_val_group_raw.shape),
    "grouped_test_shape": list(X_test_group_raw.shape),
    "train_val_group_overlap": int(len(train_groups_final & val_groups_final)),
    "train_test_group_overlap": int(len(train_groups_final & test_groups_final)),
    "val_test_group_overlap": int(len(val_groups_final & test_groups_final))
}

with open(os.path.join(OUT_DIR_2B, "companion_summary.json"), "w", encoding="utf-8") as f:
    json.dump(companion_summary, f, indent=2)

print("\n[Saved companion artifacts]")
print("Directory:", OUT_DIR_2B)
print(json.dumps(companion_summary, indent=2))

print("\nSTEP 2B completed successfully.")

In [ ]:
# STEP 2C: Leakage-resistant binary grouped split

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

print("=" * 80)
print("STEP 2C: LEAKAGE-RESISTANT BINARY GROUPED SPLIT")
print("=" * 80)

assert "df_work" in globals(), "Run Step 2 first."


# Build binary dataset after deduplication
# Keep all rows here; no rare-class filtering needed for binary
df_bin = df_work.copy().reset_index(drop=True)

df_bin["binary_target"] = pd.to_numeric(df_bin["label"], errors="coerce").fillna(-1).astype(int)

print("\n[Binary dataset shape]")
print(df_bin.shape)

print("\n[Binary target distribution]")
print(df_bin["binary_target"].value_counts())
print((100 * df_bin["binary_target"].value_counts(normalize=True)).round(4))


# Preserve context
CONTEXT_COLS_BIN = [c for c in [
    "src_ip", "dst_ip", "src_port", "dst_port", "proto", "service", "conn_state"
] if c in df_bin.columns]

print("\n[Context columns for grouped binary setup]")
print(CONTEXT_COLS_BIN)


# Feature exclusions
# Exclude raw IPs and targets from predictive features
DROP_FROM_FEATURES_BIN = [c for c in [
    "type",
    "label",
    "binary_target",
    "src_ip",
    "dst_ip"
] if c in df_bin.columns]

X_bin_raw = df_bin.drop(columns=DROP_FROM_FEATURES_BIN, errors="ignore").copy()
y_bin = df_bin["binary_target"].copy()

obj_cols_bin = X_bin_raw.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols_bin:
    X_bin_raw[c] = X_bin_raw[c].astype(str).str.strip()
    X_bin_raw[c] = X_bin_raw[c].replace({"-": "MISSING_TOKEN", "": "MISSING_TOKEN", "nan": "MISSING_TOKEN"})

print("\n[Binary raw feature matrix]")
print(X_bin_raw.shape)


# Group by directed host pair
# Stronger than random split, less degenerate than src_ip alone
assert "src_ip" in df_bin.columns and "dst_ip" in df_bin.columns, "Need src_ip and dst_ip for grouped split."

df_bin["host_pair_group"] = df_bin["src_ip"].astype(str) + "->" + df_bin["dst_ip"].astype(str)
groups_bin = df_bin["host_pair_group"].values

print("\n[Grouping metadata]")
print("Grouping column     : host_pair_group")
print("Unique host pairs   :", df_bin["host_pair_group"].nunique())


# Outer split: trainval/test
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx_b, test_idx_b = next(gss_outer.split(X_bin_raw, y_bin, groups=groups_bin))

X_trainval_bin_raw = X_bin_raw.iloc[trainval_idx_b].reset_index(drop=True)
X_test_bin_raw     = X_bin_raw.iloc[test_idx_b].reset_index(drop=True)

y_trainval_bin = y_bin.iloc[trainval_idx_b].reset_index(drop=True)
y_test_bin     = y_bin.iloc[test_idx_b].reset_index(drop=True)

groups_trainval_bin = groups_bin[trainval_idx_b]


# Inner split: train/val
gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_b2, val_idx_b2 = next(gss_inner.split(X_trainval_bin_raw, y_trainval_bin, groups=groups_trainval_bin))

X_train_bin_raw = X_trainval_bin_raw.iloc[train_idx_b2].reset_index(drop=True)
X_val_bin_raw   = X_trainval_bin_raw.iloc[val_idx_b2].reset_index(drop=True)

y_train_bin = y_trainval_bin.iloc[train_idx_b2].reset_index(drop=True)
y_val_bin   = y_trainval_bin.iloc[val_idx_b2].reset_index(drop=True)

ctx_train_bin = df_bin.iloc[np.array(trainval_idx_b)[train_idx_b2]][CONTEXT_COLS_BIN].reset_index(drop=True)
ctx_val_bin   = df_bin.iloc[np.array(trainval_idx_b)[val_idx_b2]][CONTEXT_COLS_BIN].reset_index(drop=True)
ctx_test_bin  = df_bin.iloc[test_idx_b][CONTEXT_COLS_BIN].reset_index(drop=True)

print("\n[Grouped binary split shapes]")
print("X_train_bin_raw :", X_train_bin_raw.shape)
print("X_val_bin_raw   :", X_val_bin_raw.shape)
print("X_test_bin_raw  :", X_test_bin_raw.shape)

def show_bin_dist(name, y_split):
    vc = pd.Series(y_split).value_counts().sort_index()
    pct = pd.Series(y_split).value_counts(normalize=True).sort_index() * 100
    out = pd.DataFrame({
        "class_id": vc.index,
        "class_name": ["benign" if i == 0 else "attack" for i in vc.index],
        "count": vc.values,
        "percent": pct.values.round(4)
    })
    print(f"\n[{name} binary grouped-split distribution]")
    print(out)

show_bin_dist("TRAIN", y_train_bin)
show_bin_dist("VAL", y_val_bin)
show_bin_dist("TEST", y_test_bin)


# Group overlap check
train_groups_bin = set(df_bin.iloc[np.array(trainval_idx_b)[train_idx_b2]]["host_pair_group"].astype(str).unique())
val_groups_bin   = set(df_bin.iloc[np.array(trainval_idx_b)[val_idx_b2]]["host_pair_group"].astype(str).unique())
test_groups_bin  = set(df_bin.iloc[test_idx_b]["host_pair_group"].astype(str).unique())

print("\n[Host-pair group overlap check]")
print("Train ∩ Val  :", len(train_groups_bin & val_groups_bin))
print("Train ∩ Test :", len(train_groups_bin & test_groups_bin))
print("Val ∩ Test   :", len(val_groups_bin & test_groups_bin))


# Save outputs
OUT_DIR_2C = "binary_grouped_split_outputs"
os.makedirs(OUT_DIR_2C, exist_ok=True)

step2c_summary = {
    "rows": int(len(df_bin)),
    "binary_distribution": {str(k): int(v) for k, v in df_bin["binary_target"].value_counts().sort_index().items()},
    "n_unique_host_pairs": int(df_bin["host_pair_group"].nunique()),
    "train_shape": list(X_train_bin_raw.shape),
    "val_shape": list(X_val_bin_raw.shape),
    "test_shape": list(X_test_bin_raw.shape),
    "train_val_overlap": int(len(train_groups_bin & val_groups_bin)),
    "train_test_overlap": int(len(train_groups_bin & test_groups_bin)),
    "val_test_overlap": int(len(val_groups_bin & test_groups_bin))
}

with open(os.path.join(OUT_DIR_2C, "binary_grouped_split_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step2c_summary, f, indent=2)

print("\n[Saved grouped binary artifacts]")
print("Directory:", OUT_DIR_2C)
print(json.dumps(step2c_summary, indent=2))

print("\nSTEP 2C completed successfully.")

In [ ]:

# STEP 3: Multiclass preprocessing + baseline training +
# validation-based model selection

import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("=" * 80)
print("STEP 3: MULTICLASS PREPROCESSING + TRAINING + MODEL SELECTION")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "X_train_raw", "X_val_raw", "X_test_raw",
    "y_train", "y_val", "y_test",
    "numeric_cols", "categorical_cols",
    "high_card_cat_cols", "low_mid_card_cat_cols",
    "inv_label_mapping", "label_mapping"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Step 2 first."


# Custom frequency encoder for high-cardinality categorical columns
# Learned on training only, safe on unseen values
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.freq_maps_ = {}

        n = len(X)
        for c in self.columns_:
            s = X[c].astype(str)
            vc = s.value_counts(dropna=False)
            self.freq_maps_[c] = (vc / max(n, 1)).to_dict()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = np.zeros((len(X), len(self.columns_)), dtype=float)

        for j, c in enumerate(self.columns_):
            fmap = self.freq_maps_.get(c, {})
            out[:, j] = X[c].astype(str).map(fmap).fillna(0.0).astype(float).to_numpy()

        return out


# Optional log transform for highly skewed numeric traffic features
# 
log_transform_candidates = [
    "duration", "src_bytes", "dst_bytes", "missed_bytes",
    "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes",
    "http_request_body_len", "http_response_body_len"
]
log_transform_numeric_cols = [c for c in log_transform_candidates if c in numeric_cols]
non_log_numeric_cols = [c for c in numeric_cols if c not in log_transform_numeric_cols]

print("\n[Numeric preprocessing plan]")
print("Log-transform numeric columns:")
print(log_transform_numeric_cols if log_transform_numeric_cols else "None")

print("\nNon-log numeric columns:")
print(non_log_numeric_cols if non_log_numeric_cols else "None")

print("\n[Categorical preprocessing plan]")
print("High-cardinality categorical columns:")
print(high_card_cat_cols if high_card_cat_cols else "None")

print("\nLow/mid-cardinality categorical columns:")
print(low_mid_card_cat_cols if low_mid_card_cat_cols else "None")


# Build preprocessing pipelines
# We create two preprocessors:
#   1) scaled_preprocessor  -> for logistic regression
#   2) tree_preprocessor    -> for tree-based models
# numeric branches
log_num_pipe_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, validate=False)),
    ("scaler", StandardScaler())
])

plain_num_pipe_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

log_num_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, validate=False))
])

plain_num_pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# low/mid categorical one-hot
ohe_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

# high-cardinality categorical frequency encoding
freq_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("freq", FrequencyEncoder())
])

scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("log_num", log_num_pipe_scaled, log_transform_numeric_cols),
        ("plain_num", plain_num_pipe_scaled, non_log_numeric_cols),
        ("lowmid_cat", ohe_pipe, low_mid_card_cat_cols),
        ("high_cat", freq_pipe, high_card_cat_cols),
    ],
    remainder="drop"
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("log_num", log_num_pipe_tree, log_transform_numeric_cols),
        ("plain_num", plain_num_pipe_tree, non_log_numeric_cols),
        ("lowmid_cat", ohe_pipe, low_mid_card_cat_cols),
        ("high_cat", freq_pipe, high_card_cat_cols),
    ],
    remainder="drop"
)


# Define models
models = {
    "logreg": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("clf", LogisticRegression(
            max_iter=400,
            multi_class="auto",
            class_weight="balanced",
            n_jobs=None
        ))
    ]),
    "rf": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=250,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ))
    ]),
    "extratrees": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("clf", ExtraTreesClassifier(
            n_estimators=250,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ])
}

print("\n[Candidate models]")
print(list(models.keys()))


# Evaluation helper for multiclass validation
def evaluate_multiclass_model(model, X_tr, y_tr, X_va, y_va):
    result = {}

    # fit time
    t_fit0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_s = time.perf_counter() - t_fit0

    # predict time
    t_pred0 = time.perf_counter()
    y_pred = model.predict(X_va)
    pred_s = time.perf_counter() - t_pred0

    # metrics
    acc = accuracy_score(y_va, y_pred)
    f1_macro = f1_score(y_va, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_va, y_pred, average="weighted", zero_division=0)
    prec_macro = precision_score(y_va, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_va, y_pred, average="macro", zero_division=0)

    result["val_acc"] = acc
    result["val_f1_macro"] = f1_macro
    result["val_f1_weighted"] = f1_weighted
    result["val_prec_macro"] = prec_macro
    result["val_rec_macro"] = rec_macro
    result["fit_s"] = fit_s
    result["pred_s"] = pred_s
    result["pred_ms_per_sample"] = 1000 * pred_s / len(X_va)

    # multiclass ROC-AUC (OVR macro) if predict_proba exists
    result["val_roc_auc_ovr_macro"] = np.nan
    result["val_roc_auc_ovr_weighted"] = np.nan
    result["y_pred"] = y_pred
    result["y_proba"] = None

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_va)
            result["y_proba"] = y_proba
            result["val_roc_auc_ovr_macro"] = roc_auc_score(
                y_va, y_proba, multi_class="ovr", average="macro"
            )
            result["val_roc_auc_ovr_weighted"] = roc_auc_score(
                y_va, y_proba, multi_class="ovr", average="weighted"
            )
        except Exception as e:
            print(f"ROC-AUC computation warning: {e}")

    return result


# Train/evaluate all candidate models on validation split
rows = []
val_diagnostics = {}

for name, model in models.items():
    print("\n" + "-" * 80)
    print(f"Training model: {name}")

    res = evaluate_multiclass_model(model, X_train_raw, y_train, X_val_raw, y_val)
    val_diagnostics[name] = res

    rows.append({
        "model": name,
        "val_acc": res["val_acc"],
        "val_f1_macro": res["val_f1_macro"],
        "val_f1_weighted": res["val_f1_weighted"],
        "val_prec_macro": res["val_prec_macro"],
        "val_rec_macro": res["val_rec_macro"],
        "val_roc_auc_ovr_macro": res["val_roc_auc_ovr_macro"],
        "val_roc_auc_ovr_weighted": res["val_roc_auc_ovr_weighted"],
        "fit_s": res["fit_s"],
        "pred_s": res["pred_s"],
        "pred_ms_per_sample": res["pred_ms_per_sample"]
    })

    print(f"Validation Accuracy              : {res['val_acc']:.6f}")
    print(f"Validation F1 Macro              : {res['val_f1_macro']:.6f}")
    print(f"Validation F1 Weighted           : {res['val_f1_weighted']:.6f}")
    print(f"Validation Precision Macro       : {res['val_prec_macro']:.6f}")
    print(f"Validation Recall Macro          : {res['val_rec_macro']:.6f}")
    print(f"Validation ROC-AUC OVR Macro     : {res['val_roc_auc_ovr_macro']:.6f}")
    print(f"Validation ROC-AUC OVR Weighted  : {res['val_roc_auc_ovr_weighted']:.6f}")
    print(f"Fit time (s)                     : {res['fit_s']:.4f}")
    print(f"Predict time (s)                 : {res['pred_s']:.4f}")
    print(f"Predict ms/sample                : {res['pred_ms_per_sample']:.6f}")


# Model comparison table
# Sort primarily by macro-F1 for balanced multiclass quality
scores_df = pd.DataFrame(rows).sort_values(
    by=["val_f1_macro", "val_f1_weighted", "val_roc_auc_ovr_macro", "pred_ms_per_sample"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("[VALIDATION MODEL COMPARISON TABLE]")
print("=" * 80)
print(scores_df)


# Select best model
best_name = scores_df.loc[0, "model"]
best_model = clone(models[best_name])

print("\n[Selected best model]")
print("Best model name:", best_name)


# Refit best model on TRAIN + VAL
# This is the model we will use for test evaluation in Step 4
X_trainval_raw = pd.concat([X_train_raw, X_val_raw], axis=0).reset_index(drop=True)
y_trainval = pd.concat([pd.Series(y_train), pd.Series(y_val)], axis=0).reset_index(drop=True)

t_refit0 = time.perf_counter()
best_model.fit(X_trainval_raw, y_trainval)
t_refit = time.perf_counter() - t_refit0

print("\n[Best model refit on TRAIN+VAL]")
print(f"Refit time (s): {t_refit:.4f}")


# Also fit all final models on TRAIN+VAL for later test comparison
final_models = {}
for name, model in models.items():
    m = clone(model)
    print(f"Fitting final model on TRAIN+VAL: {name}")
    m.fit(X_trainval_raw, y_trainval)
    final_models[name] = m

print("\nFinal models trained:")
print(list(final_models.keys()))


# Detailed classification report for validation best model
# Useful for interpretation before test stage
best_val_pred = val_diagnostics[best_name]["y_pred"]

print("\n" + "=" * 80)
print(f"[VALIDATION CLASSIFICATION REPORT: {best_name}]")
print("=" * 80)
print(classification_report(
    y_val,
    best_val_pred,
    target_names=[inv_label_mapping[i] for i in sorted(inv_label_mapping.keys())],
    digits=4,
    zero_division=0
))


# Validation confusion matrix
cm_val = confusion_matrix(y_val, best_val_pred, labels=sorted(inv_label_mapping.keys()))
cm_val_df = pd.DataFrame(
    cm_val,
    index=[f"true_{inv_label_mapping[i]}" for i in sorted(inv_label_mapping.keys())],
    columns=[f"pred_{inv_label_mapping[i]}" for i in sorted(inv_label_mapping.keys())]
)

print("\n[Validation confusion matrix]")
print(cm_val_df)


# Save artifacts
OUT_DIR = "multiclass_model_selection_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

scores_df.to_csv(os.path.join(OUT_DIR, "validation_model_comparison.csv"), index=False)
cm_val_df.to_csv(os.path.join(OUT_DIR, f"validation_confusion_matrix_{best_name}.csv"))

step3_summary = {
    "candidate_models": list(models.keys()),
    "best_model": best_name,
    "selection_rule": [
        "highest validation macro F1",
        "then highest validation weighted F1",
        "then highest validation OVR macro ROC-AUC",
        "then lowest latency"
    ],
    "refit_time_seconds": float(t_refit),
    "train_shape": list(X_train_raw.shape),
    "val_shape": list(X_val_raw.shape),
    "trainval_shape": list(X_trainval_raw.shape),
    "test_shape": list(X_test_raw.shape)
}

with open(os.path.join(OUT_DIR, "step3_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step3_summary, f, indent=2)

t_step3 = time.perf_counter() - t0

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Step 3 summary]")
print(json.dumps(step3_summary, indent=2))

print(f"\nSTEP 3 completed successfully in {t_step3:.4f} seconds.")

In [ ]:

# STEP 3B: Add strong tabular baseline (XGBoost / LightGBM)

import time
import json
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

print("=" * 80)
print("STEP 3B: STRONG TABULAR BASELINE")
print("=" * 80)

assert "scaled_preprocessor" in globals()
assert "tree_preprocessor" in globals()
assert "X_train_raw" in globals()
assert "X_val_raw" in globals()
assert "y_train" in globals()
assert "y_val" in globals()
assert "scores_df" in globals()

baseline_models = {}


# Try XGBoost first
xgb_available = False
lgbm_available = False

try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception as e:
    print(f"XGBoost not available: {e}")

try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception as e:
    print(f"LightGBM not available: {e}")

if xgb_available:
    baseline_models["xgboost"] = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("clf", XGBClassifier(
            n_estimators=250,
            max_depth=8,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1
        ))
    ])

elif lgbm_available:
    baseline_models["lightgbm"] = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("clf", LGBMClassifier(
            n_estimators=250,
            learning_rate=0.1,
            num_leaves=31,
            objective="multiclass",
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ])
else:
    raise ImportError("Neither XGBoost nor LightGBM is available in this environment.")

def evaluate_multiclass_model_simple(model, X_tr, y_tr, X_va, y_va):
    result = {}

    t_fit0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    result["fit_s"] = time.perf_counter() - t_fit0

    t_pred0 = time.perf_counter()
    y_pred = model.predict(X_va)
    result["pred_s"] = time.perf_counter() - t_pred0

    result["val_acc"] = accuracy_score(y_va, y_pred)
    result["val_f1_macro"] = f1_score(y_va, y_pred, average="macro", zero_division=0)
    result["val_f1_weighted"] = f1_score(y_va, y_pred, average="weighted", zero_division=0)
    result["val_prec_macro"] = precision_score(y_va, y_pred, average="macro", zero_division=0)
    result["val_rec_macro"] = recall_score(y_va, y_pred, average="macro", zero_division=0)
    result["pred_ms_per_sample"] = 1000 * result["pred_s"] / len(X_va)

    result["val_roc_auc_ovr_macro"] = np.nan
    result["val_roc_auc_ovr_weighted"] = np.nan

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_va)
        result["val_roc_auc_ovr_macro"] = roc_auc_score(
            y_va, y_proba, multi_class="ovr", average="macro"
        )
        result["val_roc_auc_ovr_weighted"] = roc_auc_score(
            y_va, y_proba, multi_class="ovr", average="weighted"
        )

    return result

new_rows = []

for name, model in baseline_models.items():
    print("\n" + "-" * 80)
    print(f"Training strong baseline: {name}")

    res = evaluate_multiclass_model_simple(model, X_train_raw, y_train, X_val_raw, y_val)

    new_rows.append({
        "model": name,
        "val_acc": res["val_acc"],
        "val_f1_macro": res["val_f1_macro"],
        "val_f1_weighted": res["val_f1_weighted"],
        "val_prec_macro": res["val_prec_macro"],
        "val_rec_macro": res["val_rec_macro"],
        "val_roc_auc_ovr_macro": res["val_roc_auc_ovr_macro"],
        "val_roc_auc_ovr_weighted": res["val_roc_auc_ovr_weighted"],
        "fit_s": res["fit_s"],
        "pred_s": res["pred_s"],
        "pred_ms_per_sample": res["pred_ms_per_sample"]
    })

    print(f"Validation Accuracy              : {res['val_acc']:.6f}")
    print(f"Validation F1 Macro              : {res['val_f1_macro']:.6f}")
    print(f"Validation F1 Weighted           : {res['val_f1_weighted']:.6f}")
    print(f"Validation Precision Macro       : {res['val_prec_macro']:.6f}")
    print(f"Validation Recall Macro          : {res['val_rec_macro']:.6f}")
    print(f"Validation ROC-AUC OVR Macro     : {res['val_roc_auc_ovr_macro']:.6f}")
    print(f"Validation ROC-AUC OVR Weighted  : {res['val_roc_auc_ovr_weighted']:.6f}")
    print(f"Fit time (s)                     : {res['fit_s']:.4f}")
    print(f"Predict ms/sample                : {res['pred_ms_per_sample']:.6f}")

scores_df_extended = pd.concat([scores_df, pd.DataFrame(new_rows)], axis=0)
scores_df_extended = scores_df_extended.sort_values(
    by=["val_f1_macro", "val_f1_weighted", "val_roc_auc_ovr_macro", "pred_ms_per_sample"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("[EXTENDED VALIDATION MODEL COMPARISON TABLE]")
print("=" * 80)
print(scores_df_extended)

OUT_DIR = "multiclass_model_selection_outputs"
scores_df_extended.to_csv(os.path.join(OUT_DIR, "validation_model_comparison_extended.csv"), index=False)

print("\nSTEP 3B completed successfully.")

In [ ]:
# STEP 3C: Add LightGBM baseline and update extended comparison

import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

print("=" * 80)
print("STEP 3C: LIGHTGBM BASELINE")
print("=" * 80)


# Safety checks
required_vars = [
    "tree_preprocessor",
    "X_train_raw", "X_val_raw",
    "y_train", "y_val",
    "scores_df_extended"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run previous steps first."

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError(
        f"LightGBM is not available in this environment: {e}\n"
        "Install it first or skip this step."
    )


# Evaluation helper
def evaluate_multiclass_model_simple(model, X_tr, y_tr, X_va, y_va):
    result = {}

    t_fit0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    result["fit_s"] = time.perf_counter() - t_fit0

    t_pred0 = time.perf_counter()
    y_pred = model.predict(X_va)
    result["pred_s"] = time.perf_counter() - t_pred0

    result["val_acc"] = accuracy_score(y_va, y_pred)
    result["val_f1_macro"] = f1_score(y_va, y_pred, average="macro", zero_division=0)
    result["val_f1_weighted"] = f1_score(y_va, y_pred, average="weighted", zero_division=0)
    result["val_prec_macro"] = precision_score(y_va, y_pred, average="macro", zero_division=0)
    result["val_rec_macro"] = recall_score(y_va, y_pred, average="macro", zero_division=0)
    result["pred_ms_per_sample"] = 1000 * result["pred_s"] / len(X_va)

    result["val_roc_auc_ovr_macro"] = np.nan
    result["val_roc_auc_ovr_weighted"] = np.nan
    result["y_pred"] = y_pred
    result["y_proba"] = None

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_va)
            result["y_proba"] = y_proba
            result["val_roc_auc_ovr_macro"] = roc_auc_score(
                y_va, y_proba, multi_class="ovr", average="macro"
            )
            result["val_roc_auc_ovr_weighted"] = roc_auc_score(
                y_va, y_proba, multi_class="ovr", average="weighted"
            )
        except Exception as e:
            print(f"ROC-AUC computation warning: {e}")

    return result


# Define LightGBM model
lightgbm_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("clf", LGBMClassifier(
        objective="multiclass",
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
        verbosity=-1
    ))
])

print("\n[Training strong baseline: lightgbm]")
res_lgbm = evaluate_multiclass_model_simple(
    lightgbm_model, X_train_raw, y_train, X_val_raw, y_val
)

print(f"Validation Accuracy              : {res_lgbm['val_acc']:.6f}")
print(f"Validation F1 Macro              : {res_lgbm['val_f1_macro']:.6f}")
print(f"Validation F1 Weighted           : {res_lgbm['val_f1_weighted']:.6f}")
print(f"Validation Precision Macro       : {res_lgbm['val_prec_macro']:.6f}")
print(f"Validation Recall Macro          : {res_lgbm['val_rec_macro']:.6f}")
print(f"Validation ROC-AUC OVR Macro     : {res_lgbm['val_roc_auc_ovr_macro']:.6f}")
print(f"Validation ROC-AUC OVR Weighted  : {res_lgbm['val_roc_auc_ovr_weighted']:.6f}")
print(f"Fit time (s)                     : {res_lgbm['fit_s']:.4f}")
print(f"Predict time (s)                 : {res_lgbm['pred_s']:.4f}")
print(f"Predict ms/sample                : {res_lgbm['pred_ms_per_sample']:.6f}")


# Store in baseline_models if present, otherwise create it
if "baseline_models" not in globals():
    baseline_models = {}

baseline_models["lightgbm"] = lightgbm_model


# Update extended comparison table
lgbm_row = pd.DataFrame([{
    "model": "lightgbm",
    "val_acc": res_lgbm["val_acc"],
    "val_f1_macro": res_lgbm["val_f1_macro"],
    "val_f1_weighted": res_lgbm["val_f1_weighted"],
    "val_prec_macro": res_lgbm["val_prec_macro"],
    "val_rec_macro": res_lgbm["val_rec_macro"],
    "val_roc_auc_ovr_macro": res_lgbm["val_roc_auc_ovr_macro"],
    "val_roc_auc_ovr_weighted": res_lgbm["val_roc_auc_ovr_weighted"],
    "fit_s": res_lgbm["fit_s"],
    "pred_s": res_lgbm["pred_s"],
    "pred_ms_per_sample": res_lgbm["pred_ms_per_sample"]
}])

# avoid duplicate row if rerun
scores_df_extended_no_lgbm = scores_df_extended[scores_df_extended["model"] != "lightgbm"].copy()

scores_df_extended = pd.concat(
    [scores_df_extended_no_lgbm, lgbm_row],
    axis=0,
    ignore_index=True
).sort_values(
    by=["val_f1_macro", "val_f1_weighted", "val_roc_auc_ovr_macro", "pred_ms_per_sample"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("[UPDATED EXTENDED VALIDATION MODEL COMPARISON TABLE]")
print("=" * 80)
print(scores_df_extended)


# Save updated comparison
OUT_DIR = "multiclass_model_selection_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

scores_df_extended.to_csv(
    os.path.join(OUT_DIR, "validation_model_comparison_extended_with_lightgbm.csv"),
    index=False
)

step3c_lgbm_summary = {
    "lightgbm_metrics": {
        "val_acc": float(res_lgbm["val_acc"]),
        "val_f1_macro": float(res_lgbm["val_f1_macro"]),
        "val_f1_weighted": float(res_lgbm["val_f1_weighted"]),
        "val_prec_macro": float(res_lgbm["val_prec_macro"]),
        "val_rec_macro": float(res_lgbm["val_rec_macro"]),
        "val_roc_auc_ovr_macro": float(res_lgbm["val_roc_auc_ovr_macro"]),
        "val_roc_auc_ovr_weighted": float(res_lgbm["val_roc_auc_ovr_weighted"]),
        "fit_s": float(res_lgbm["fit_s"]),
        "pred_s": float(res_lgbm["pred_s"]),
        "pred_ms_per_sample": float(res_lgbm["pred_ms_per_sample"])
    },
    "current_best_model": str(scores_df_extended.loc[0, "model"]),
    "models_in_extended_table": scores_df_extended["model"].tolist()
}

with open(os.path.join(OUT_DIR, "step3c_lightgbm_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step3c_lgbm_summary, f, indent=2)

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)
print(json.dumps(step3c_lgbm_summary, indent=2))

print("\nSTEP 3C completed successfully.")

In [ ]:
# STEP 3D: Finalize best multiclass model after all baselines

import os
import json
import time
import pandas as pd
from sklearn.base import clone

print("=" * 80)
print("STEP 3D: FINALIZE BEST MULTICLASS MODEL")
print("=" * 80)

required_vars = [
    "scores_df_extended",
    "models",
    "baseline_models",
    "X_train_raw", "X_val_raw", "X_test_raw",
    "y_train", "y_val"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

# merge baseline models into main model pool
for k, v in baseline_models.items():
    models[k] = v

best_name_extended = scores_df_extended.loc[0, "model"]
best_model = clone(models[best_name_extended])

print("\n[Selected best model after full comparison]")
print("Best model name:", best_name_extended)

X_trainval_raw = pd.concat([X_train_raw, X_val_raw], axis=0).reset_index(drop=True)
y_trainval = pd.concat([pd.Series(y_train), pd.Series(y_val)], axis=0).reset_index(drop=True)

t0 = time.perf_counter()
best_model.fit(X_trainval_raw, y_trainval)
refit_time_extended = time.perf_counter() - t0

print(f"Refit time on TRAIN+VAL (s): {refit_time_extended:.4f}")

final_models = {}
for name, model in models.items():
    m = clone(model)
    print(f"Fitting final model on TRAIN+VAL: {name}")
    m.fit(X_trainval_raw, y_trainval)
    final_models[name] = m

print("\n[Final model pool]")
print(list(final_models.keys()))

OUT_DIR = "multiclass_model_selection_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

step3d_summary = {
    "best_model_after_full_comparison": best_name_extended,
    "candidate_models_final": list(final_models.keys()),
    "refit_time_seconds": float(refit_time_extended),
    "trainval_shape": list(X_trainval_raw.shape),
    "test_shape": list(X_test_raw.shape)
}

with open(os.path.join(OUT_DIR, "step3d_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step3d_summary, f, indent=2)

print("\n[Saved updated model-selection summary]")
print(json.dumps(step3d_summary, indent=2))

print("\nSTEP 3D completed successfully.")

In [ ]:
# STEP 4: Multiclass test evaluation + comparison of final models


import os
import json
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    top_k_accuracy_score
)

print("=" * 80)
print("STEP 4: MULTICLASS TEST EVALUATION + FINAL MODEL COMPARISON")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "best_model", "final_models",
    "X_test_raw", "y_test", "inv_label_mapping"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Step 3 first."

# final best model name after all baselines
if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

class_ids = sorted(inv_label_mapping.keys())
class_names = [inv_label_mapping[i] for i in class_ids]

print("\n[Best model entering test evaluation]")
print("Best model name:", best_name_final)

print("\n[Test set shape]")
print(X_test_raw.shape)

print("\n[Test class names]")
print(class_names)


# Helper: evaluate a multiclass model on test
def evaluate_multiclass_test(model, X, y, class_ids):
    result = {}

    # predict labels
    t_pred0 = time.perf_counter()
    y_pred = model.predict(X)
    pred_s = time.perf_counter() - t_pred0

    result["y_pred"] = y_pred
    result["pred_time_s"] = pred_s
    result["pred_ms_per_sample"] = 1000 * pred_s / len(X)

    # basic metrics
    result["test_acc"] = accuracy_score(y, y_pred)
    result["test_f1_macro"] = f1_score(y, y_pred, average="macro", zero_division=0)
    result["test_f1_weighted"] = f1_score(y, y_pred, average="weighted", zero_division=0)
    result["test_prec_macro"] = precision_score(y, y_pred, average="macro", zero_division=0)
    result["test_rec_macro"] = recall_score(y, y_pred, average="macro", zero_division=0)

    # confusion matrix
    result["cm"] = confusion_matrix(y, y_pred, labels=class_ids)

    # probability-based metrics
    result["y_proba"] = None
    result["proba_time_s"] = np.nan
    result["proba_ms_per_sample"] = np.nan
    result["test_roc_auc_ovr_macro"] = np.nan
    result["test_roc_auc_ovr_weighted"] = np.nan
    result["top2_acc"] = np.nan
    result["top3_acc"] = np.nan

    if hasattr(model, "predict_proba"):
        try:
            t_proba0 = time.perf_counter()
            y_proba = model.predict_proba(X)
            proba_s = time.perf_counter() - t_proba0

            result["y_proba"] = y_proba
            result["proba_time_s"] = proba_s
            result["proba_ms_per_sample"] = 1000 * proba_s / len(X)

            result["test_roc_auc_ovr_macro"] = roc_auc_score(
                y, y_proba, multi_class="ovr", average="macro"
            )
            result["test_roc_auc_ovr_weighted"] = roc_auc_score(
                y, y_proba, multi_class="ovr", average="weighted"
            )

            result["top2_acc"] = top_k_accuracy_score(y, y_proba, k=2, labels=class_ids)
            result["top3_acc"] = top_k_accuracy_score(y, y_proba, k=3, labels=class_ids)

        except Exception as e:
            print(f"Probability-based metric warning: {e}")

    return result


# Evaluate best model on test
best_test_res = evaluate_multiclass_test(best_model, X_test_raw, y_test, class_ids)

print("\n" + "=" * 80)
print(f"[TEST RESULTS: BEST MODEL = {best_name_final}]")
print("=" * 80)
print(f"Test Accuracy                 : {best_test_res['test_acc']:.6f}")
print(f"Test F1 Macro                 : {best_test_res['test_f1_macro']:.6f}")
print(f"Test F1 Weighted              : {best_test_res['test_f1_weighted']:.6f}")
print(f"Test Precision Macro          : {best_test_res['test_prec_macro']:.6f}")
print(f"Test Recall Macro             : {best_test_res['test_rec_macro']:.6f}")
print(f"Test ROC-AUC OVR Macro        : {best_test_res['test_roc_auc_ovr_macro']:.6f}")
print(f"Test ROC-AUC OVR Weighted     : {best_test_res['test_roc_auc_ovr_weighted']:.6f}")
print(f"Top-2 Accuracy                : {best_test_res['top2_acc']:.6f}")
print(f"Top-3 Accuracy                : {best_test_res['top3_acc']:.6f}")
print(f"Predict time total (s)        : {best_test_res['pred_time_s']:.6f}")
print(f"Predict latency (ms/sample)   : {best_test_res['pred_ms_per_sample']:.6f}")
print(f"Predict_proba total (s)       : {best_test_res['proba_time_s']:.6f}")
print(f"Predict_proba (ms/sample)     : {best_test_res['proba_ms_per_sample']:.6f}")


# Best model classification report
print("\n" + "=" * 80)
print(f"[TEST CLASSIFICATION REPORT: {best_name_final}]")
print("=" * 80)
print(classification_report(
    y_test,
    best_test_res["y_pred"],
    target_names=class_names,
    digits=4,
    zero_division=0
))


# Best model confusion matrix
cm_test = best_test_res["cm"]
cm_test_df = pd.DataFrame(
    cm_test,
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names]
)

print("\n[Best model test confusion matrix]")
print(cm_test_df)


# Per-class metrics table
report_dict = classification_report(
    y_test,
    best_test_res["y_pred"],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

per_class_rows = []
for cname in class_names:
    per_class_rows.append({
        "class_name": cname,
        "precision": report_dict[cname]["precision"],
        "recall": report_dict[cname]["recall"],
        "f1_score": report_dict[cname]["f1-score"],
        "support": report_dict[cname]["support"]
    })

per_class_df = pd.DataFrame(per_class_rows).sort_values(by="f1_score", ascending=False).reset_index(drop=True)

print("\n[Per-class performance table]")
print(per_class_df)


# Compare all final models on test
compare_rows = []

for name, model in final_models.items():
    print(f"\nEvaluating final model on test: {name}")
    res = evaluate_multiclass_test(model, X_test_raw, y_test, class_ids)

    compare_rows.append({
        "model": name,
        "test_acc": res["test_acc"],
        "test_f1_macro": res["test_f1_macro"],
        "test_f1_weighted": res["test_f1_weighted"],
        "test_prec_macro": res["test_prec_macro"],
        "test_rec_macro": res["test_rec_macro"],
        "test_roc_auc_ovr_macro": res["test_roc_auc_ovr_macro"],
        "test_roc_auc_ovr_weighted": res["test_roc_auc_ovr_weighted"],
        "top2_acc": res["top2_acc"],
        "top3_acc": res["top3_acc"],
        "pred_time_s": res["pred_time_s"],
        "pred_ms_per_sample": res["pred_ms_per_sample"],
        "proba_time_s": res["proba_time_s"],
        "proba_ms_per_sample": res["proba_ms_per_sample"]
    })

compare_test_df = pd.DataFrame(compare_rows).sort_values(
    by=["test_f1_macro", "test_f1_weighted", "test_roc_auc_ovr_macro", "pred_ms_per_sample"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("[TEST MODEL COMPARISON TABLE]")
print("=" * 80)
print(compare_test_df)


# Compact summary 
summary_table = compare_test_df[[
    "model",
    "test_acc",
    "test_f1_macro",
    "test_f1_weighted",
    "test_roc_auc_ovr_macro",
    "pred_ms_per_sample"
]].copy()

print("\n[Compact ready summary table]")
print(summary_table)


# Hardest classes analysis
hardest_classes_df = per_class_df.sort_values(by="f1_score", ascending=True).reset_index(drop=True)

print("\n[Hardest classes by F1]")
print(hardest_classes_df)


# Save artifacts
OUT_DIR = "multiclass_test_evaluation_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

cm_test_df.to_csv(os.path.join(OUT_DIR, f"test_confusion_matrix_{best_name_final}.csv"))
per_class_df.to_csv(os.path.join(OUT_DIR, f"test_per_class_metrics_{best_name_final}.csv"), index=False)
compare_test_df.to_csv(os.path.join(OUT_DIR, "test_model_comparison.csv"), index=False)
summary_table.to_csv(os.path.join(OUT_DIR, "test_summary_table.csv"), index=False)
hardest_classes_df.to_csv(os.path.join(OUT_DIR, f"hardest_classes_{best_name_final}.csv"), index=False)

step4_summary = {
    "best_model": best_name_final,
    "test_shape": list(X_test_raw.shape),
    "class_names": class_names,
    "best_model_test_metrics": {
        "accuracy": float(best_test_res["test_acc"]),
        "f1_macro": float(best_test_res["test_f1_macro"]),
        "f1_weighted": float(best_test_res["test_f1_weighted"]),
        "precision_macro": float(best_test_res["test_prec_macro"]),
        "recall_macro": float(best_test_res["test_rec_macro"]),
        "roc_auc_ovr_macro": float(best_test_res["test_roc_auc_ovr_macro"]),
        "roc_auc_ovr_weighted": float(best_test_res["test_roc_auc_ovr_weighted"]),
        "top2_accuracy": float(best_test_res["top2_acc"]),
        "top3_accuracy": float(best_test_res["top3_acc"]),
        "pred_ms_per_sample": float(best_test_res["pred_ms_per_sample"]),
        "proba_ms_per_sample": float(best_test_res["proba_ms_per_sample"])
    }
}

with open(os.path.join(OUT_DIR, "step4_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step4_summary, f, indent=2)

t_step4 = time.perf_counter() - t0

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Step 4 summary]")
print(json.dumps(step4_summary, indent=2))

print(f"\nSTEP 4 completed successfully in {t_step4:.4f} seconds.")

In [ ]:

# STEP 5A: Multiclass global explainability
# - built-in importance when available
# - permutation importance
# - transformed feature-name recovery
# - final-model compatible (LightGBM / XGBoost / RF / ExtraTrees / LogReg)

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance

print("=" * 80)
print("STEP 5A: MULTICLASS GLOBAL EXPLAINABILITY")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "best_model",
    "X_trainval_raw", "X_test_raw", "y_test",
    "numeric_cols", "log_transform_numeric_cols", "non_log_numeric_cols",
    "low_mid_card_cat_cols", "high_card_cat_cols"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Steps 2-4 first."

if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

print("\n[Best model for XAI]")
print("Model:", best_name_final)


# Get fitted pipeline parts
preprocessor = best_model.named_steps["preprocessor"]
clf = best_model.named_steps["clf"]


# Recover transformed feature names

def get_transformed_feature_names(preprocessor, low_mid_cat_cols, high_card_cat_cols,
                                  log_num_cols, plain_num_cols):
    feature_names = []

    # 1) log numeric columns
    feature_names.extend([f"{c}__log1p" for c in log_num_cols])

    # 2) plain numeric columns
    feature_names.extend([f"{c}__scaled" for c in plain_num_cols])

    # 3) one-hot categorical columns
    if len(low_mid_cat_cols) > 0:
        ohe = preprocessor.named_transformers_["lowmid_cat"].named_steps["onehot"]
        ohe_names = ohe.get_feature_names_out(low_mid_cat_cols).tolist()
        feature_names.extend(ohe_names)

    # 4) high-cardinality frequency encoded columns
    if len(high_card_cat_cols) > 0:
        feature_names.extend([f"{c}__freq" for c in high_card_cat_cols])

    return feature_names

transformed_feature_names = get_transformed_feature_names(
    preprocessor=preprocessor,
    low_mid_cat_cols=low_mid_card_cat_cols,
    high_card_cat_cols=high_card_cat_cols,
    log_num_cols=log_transform_numeric_cols,
    plain_num_cols=non_log_numeric_cols
)

print("\n[Recovered transformed feature space]")
print("Number of transformed features:", len(transformed_feature_names))
print("First 50 transformed feature names:")
print(transformed_feature_names[:50])


# Built-in model importance
# Supports:
# - tree models via feature_importances_
# - logistic regression via absolute coefficients

builtin_importance_df = None

if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
    builtin_importance_df = pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": importances
    }).sort_values(by="importance", ascending=False).reset_index(drop=True)

    print("\n[Top 25 built-in global feature importances]")
    print(builtin_importance_df.head(25))

elif hasattr(clf, "coef_"):
    coef = clf.coef_
    if coef.ndim == 2:
        # multiclass: aggregate absolute coefficients across classes
        coef_abs = np.mean(np.abs(coef), axis=0)
    else:
        coef_abs = np.abs(coef)

    builtin_importance_df = pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": coef_abs
    }).sort_values(by="importance", ascending=False).reset_index(drop=True)

    print("\n[Top 25 coefficient-based global importances]")
    print(builtin_importance_df.head(25))

else:
    print("\nBuilt-in feature importance is not available for this model.")


# Aggregate transformed importances back to original raw features
def map_transformed_to_raw_feature_name(fname):
    if fname.endswith("__log1p"):
        return fname.replace("__log1p", "")
    if fname.endswith("__scaled"):
        return fname.replace("__scaled", "")
    if fname.endswith("__freq"):
        return fname.replace("__freq", "")

    # one-hot names
    all_candidate_prefixes = low_mid_card_cat_cols
    for c in sorted(all_candidate_prefixes, key=len, reverse=True):
        if fname.startswith(c + "_"):
            return c
    return fname

aggregated_builtin_df = None
if builtin_importance_df is not None:
    aggregated_builtin_df = builtin_importance_df.copy()
    aggregated_builtin_df["raw_feature"] = aggregated_builtin_df["feature"].apply(map_transformed_to_raw_feature_name)
    aggregated_builtin_df = aggregated_builtin_df.groupby("raw_feature", as_index=False)["importance"].sum()
    aggregated_builtin_df = aggregated_builtin_df.sort_values(by="importance", ascending=False).reset_index(drop=True)

    print("\n[Top 25 aggregated raw-feature importances]")
    print(aggregated_builtin_df.head(25))


# Permutation importance on held-out test subset
# Use macro-F1 because this is multiclass and class balance matters
MAX_PERM_SAMPLES = min(12000, len(X_test_raw))
X_perm = X_test_raw.iloc[:MAX_PERM_SAMPLES].copy()
y_perm = y_test.iloc[:MAX_PERM_SAMPLES].copy() if isinstance(y_test, pd.Series) else pd.Series(y_test[:MAX_PERM_SAMPLES])

print("\n[Permutation importance setup]")
print("Samples used:", len(X_perm))
print("Scoring:", "f1_macro")
print("Repeats:", 5)

t_perm0 = time.perf_counter()
perm = permutation_importance(
    best_model,
    X_perm,
    y_perm,
    scoring="f1_macro",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)
t_perm = time.perf_counter() - t_perm0

perm_importance_df = pd.DataFrame({
    "raw_feature": X_perm.columns,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std": perm.importances_std
}).sort_values(by="perm_importance_mean", ascending=False).reset_index(drop=True)

print("\n[Top 25 permutation importances]")
print(perm_importance_df.head(25))
print(f"\nPermutation importance time (s): {t_perm:.4f}")


# Plot top features
TOPK = 20

if aggregated_builtin_df is not None:
    top_builtin = aggregated_builtin_df.head(TOPK)

    plt.figure(figsize=(10, 6))
    plt.barh(top_builtin["raw_feature"][::-1], top_builtin["importance"][::-1])
    plt.title(f"Top-{TOPK} Global Feature Importance ({best_name_final}, aggregated)")
    plt.xlabel("Importance")
    plt.ylabel("Raw feature")
    plt.tight_layout()
    plt.show()

top_perm = perm_importance_df.head(TOPK)
plt.figure(figsize=(10, 6))
plt.barh(top_perm["raw_feature"][::-1], top_perm["perm_importance_mean"][::-1])
plt.title(f"Top-{TOPK} Permutation Importance ({best_name_final}, macro-F1)")
plt.xlabel("Mean importance decrease")
plt.ylabel("Raw feature")
plt.tight_layout()
plt.show()


# Consensus importance table
if aggregated_builtin_df is not None:
    xai_consensus_df = aggregated_builtin_df.merge(
        perm_importance_df,
        on="raw_feature",
        how="outer"
    ).fillna(0)

    xai_consensus_df["builtin_rank"] = xai_consensus_df["importance"].rank(ascending=False, method="min")
    xai_consensus_df["perm_rank"] = xai_consensus_df["perm_importance_mean"].rank(ascending=False, method="min")
    xai_consensus_df["rank_sum"] = xai_consensus_df["builtin_rank"] + xai_consensus_df["perm_rank"]

    xai_consensus_df = xai_consensus_df.sort_values(
        by=["rank_sum", "importance", "perm_importance_mean"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    print("\n[Top 25 consensus-important raw features]")
    print(xai_consensus_df.head(25))
else:
    xai_consensus_df = perm_importance_df.copy()


# Save artifacts
OUT_DIR = "multiclass_global_xai_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

if builtin_importance_df is not None:
    builtin_importance_df.to_csv(os.path.join(OUT_DIR, "builtin_transformed_feature_importance.csv"), index=False)

if aggregated_builtin_df is not None:
    aggregated_builtin_df.to_csv(os.path.join(OUT_DIR, "builtin_raw_feature_importance.csv"), index=False)

perm_importance_df.to_csv(os.path.join(OUT_DIR, "permutation_importance.csv"), index=False)
xai_consensus_df.to_csv(os.path.join(OUT_DIR, "xai_consensus_importance.csv"), index=False)

step5a_summary = {
    "best_model": best_name_final,
    "n_transformed_features": int(len(transformed_feature_names)),
    "permutation_samples_used": int(len(X_perm)),
    "permutation_scoring": "f1_macro",
    "permutation_repeats": 5,
    "permutation_time_seconds": float(t_perm)
}

with open(os.path.join(OUT_DIR, "step5a_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step5a_summary, f, indent=2)

t_step5a = time.perf_counter() - t0

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Step 5A summary]")
print(json.dumps(step5a_summary, indent=2))

print(f"\nSTEP 5A completed successfully in {t_step5a:.4f} seconds.")

In [ ]:
# STEP 5B: Multiclass SHAP analysis for selected final tree model
# - computation-safe version
# - supports LightGBM / XGBoost / RF / ExtraTrees


import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

print("=" * 80)
print("STEP 5B: MULTICLASS SHAP ANALYSIS")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "best_model",
    "X_test_raw", "y_test",
    "inv_label_mapping",
    "transformed_feature_names",
    "low_mid_card_cat_cols"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Step 5A first."

if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

if best_name_final not in ["lightgbm", "xgboost", "rf", "extratrees"]:
    raise ValueError(f"SHAP block currently expects a tree-based final model, got: {best_name_final}")

print("\n[Model for SHAP]")
print("Best model:", best_name_final)


# Get fitted preprocessor and classifier
preprocessor = best_model.named_steps["preprocessor"]
clf = best_model.named_steps["clf"]


# Small sample for SHAP to keep runtime stable
MAX_SHAP_SAMPLES = min(300, len(X_test_raw))
X_shap_raw = X_test_raw.iloc[:MAX_SHAP_SAMPLES].copy()
y_shap = y_test.iloc[:MAX_SHAP_SAMPLES].copy() if isinstance(y_test, pd.Series) else pd.Series(y_test[:MAX_SHAP_SAMPLES])

print("\n[SHAP sample setup]")
print("Raw sample size:", len(X_shap_raw))

# transform using the fitted preprocessor
X_shap_trans = preprocessor.transform(X_shap_raw)

# convert sparse to dense if needed
if hasattr(X_shap_trans, "toarray"):
    X_shap_trans = X_shap_trans.toarray()

X_shap_trans = np.asarray(X_shap_trans)

print("Transformed SHAP matrix shape:", X_shap_trans.shape)
print("Number of transformed feature names:", len(transformed_feature_names))

assert X_shap_trans.shape[1] == len(transformed_feature_names), \
    "Mismatch between transformed matrix width and transformed feature names."


# SHAP computation
t_shap0 = time.perf_counter()

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_shap_trans)

t_shap = time.perf_counter() - t_shap0

print(f"\nSHAP computation time (s): {t_shap:.4f}")


# Normalize SHAP output shape
# Different SHAP versions may return:
# - list of arrays, one per class
# - single 3D array (samples, features, classes)
if isinstance(shap_values, list):
    shap_by_class = [np.asarray(sv) for sv in shap_values]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_by_class = [shap_values[:, :, k] for k in range(shap_values.shape[2])]
else:
    raise ValueError(
        f"Unexpected SHAP output structure: type={type(shap_values)}, "
        f"shape={getattr(shap_values, 'shape', None)}"
    )

n_classes = len(shap_by_class)
print("Detected SHAP class count:", n_classes)

class_names = [inv_label_mapping[i] for i in sorted(inv_label_mapping.keys())]
assert n_classes == len(class_names), \
    f"SHAP class count {n_classes} != label class count {len(class_names)}"


# Global mean absolute SHAP aggregated across all classes
mean_abs_shap_per_class = []
for k, sv in enumerate(shap_by_class):
    mean_abs = np.mean(np.abs(sv), axis=0)
    mean_abs_shap_per_class.append(mean_abs)

mean_abs_shap_per_class = np.vstack(mean_abs_shap_per_class)  # (n_classes, n_features)
global_mean_abs_shap = mean_abs_shap_per_class.mean(axis=0)

global_shap_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "mean_abs_shap": global_mean_abs_shap
}).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n[Top 25 transformed features by global mean |SHAP|]")
print(global_shap_df.head(25))


# Aggregate transformed SHAP back to raw feature level
def map_transformed_to_raw_feature_name(fname):
    if fname.endswith("__log1p"):
        return fname.replace("__log1p", "")
    if fname.endswith("__scaled"):
        return fname.replace("__scaled", "")
    if fname.endswith("__freq"):
        return fname.replace("__freq", "")

    for c in sorted(low_mid_card_cat_cols, key=len, reverse=True):
        if fname.startswith(c + "_"):
            return c
    return fname

global_shap_raw_df = global_shap_df.copy()
global_shap_raw_df["raw_feature"] = global_shap_raw_df["feature"].apply(map_transformed_to_raw_feature_name)
global_shap_raw_df = global_shap_raw_df.groupby("raw_feature", as_index=False)["mean_abs_shap"].sum()
global_shap_raw_df = global_shap_raw_df.sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n[Top 25 raw features by aggregated global mean |SHAP|]")
print(global_shap_raw_df.head(25))


# Per-class SHAP tables
per_class_shap_tables = {}

for k, cname in enumerate(class_names):
    class_df = pd.DataFrame({
        "feature": transformed_feature_names,
        "mean_abs_shap": mean_abs_shap_per_class[k]
    }).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

    class_df["raw_feature"] = class_df["feature"].apply(map_transformed_to_raw_feature_name)
    class_raw_df = class_df.groupby("raw_feature", as_index=False)["mean_abs_shap"].sum()
    class_raw_df = class_raw_df.sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

    per_class_shap_tables[cname] = class_raw_df

    print("\n" + "-" * 80)
    print(f"[Top 10 raw SHAP features for class: {cname}]")
    print(class_raw_df.head(10))


# Plot global raw SHAP top features
TOPK = 20
top_global_raw = global_shap_raw_df.head(TOPK)

plt.figure(figsize=(10, 6))
plt.barh(top_global_raw["raw_feature"][::-1], top_global_raw["mean_abs_shap"][::-1])
plt.title(f"Top-{TOPK} Global SHAP Features ({best_name_final}, aggregated raw features)")
plt.xlabel("Mean |SHAP value|")
plt.ylabel("Raw feature")
plt.tight_layout()
plt.show()


# Representative summary plot
# choose one difficult class if available
preferred_class = "xss" if "xss" in class_names else class_names[0]
preferred_idx = class_names.index(preferred_class)

print("\n[Representative class summary plot]")
print("Chosen class:", preferred_class)

try:
    shap.summary_plot(
        shap_by_class[preferred_idx],
        X_shap_trans,
        feature_names=transformed_feature_names,
        show=True,
        max_display=20
    )
except Exception as e:
    print("SHAP summary plot warning:", e)


# Save artifacts
OUT_DIR = "multiclass_shap_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

global_shap_df.to_csv(os.path.join(OUT_DIR, "global_transformed_shap_importance.csv"), index=False)
global_shap_raw_df.to_csv(os.path.join(OUT_DIR, "global_raw_shap_importance.csv"), index=False)

for cname, class_df in per_class_shap_tables.items():
    safe_name = cname.replace("/", "_").replace(" ", "_")
    class_df.to_csv(os.path.join(OUT_DIR, f"shap_raw_importance_{safe_name}.csv"), index=False)

step5b_summary = {
    "best_model": best_name_final,
    "shap_sample_size": int(len(X_shap_raw)),
    "transformed_feature_count": int(X_shap_trans.shape[1]),
    "class_count": int(n_classes),
    "class_names": class_names,
    "representative_class_plot": preferred_class,
    "shap_time_seconds": float(t_shap)
}

with open(os.path.join(OUT_DIR, "step5b_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step5b_summary, f, indent=2)

t_step5b = time.perf_counter() - t0

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Step 5B summary]")
print(json.dumps(step5b_summary, indent=2))

print(f"\nSTEP 5B completed successfully in {t_step5b:.4f} seconds.")

In [ ]:
# STEP 6A: Multiclass forensic evidence logging
# - append-only SHA-256 hash-chained evidence records
# - alert = predicted class != normal
# - stores predicted class, confidence, top-k probabilities
# - stores per-alert signed SHAP attributions


import os
import json
import time
import hashlib
import datetime
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict

print("=" * 80)
print("STEP 6A: MULTICLASS FORENSIC EVIDENCE LOGGING")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "best_model",
    "X_test_raw", "y_test", "ctx_test",
    "inv_label_mapping", "label_mapping",
    "preprocessor", "transformed_feature_names",
    "shap_by_class", "X_shap_raw", "X_shap_trans"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Steps 5A-5B first."

if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

assert "normal" in label_mapping, "'normal' class not found in label mapping."
NORMAL_CLASS_ID = label_mapping["normal"]


# Hash helpers
HASH_ALGORITHM = "SHA-256"

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def stable_hash(obj) -> str:
    s = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return sha256_hex(s)


# Evidence schema
@dataclass
class MultiClassEvidenceRecord:
    record_id: str
    created_utc: str
    case_id: str
    event_time: str
    context: dict

    model_name: str
    model_digest: str
    feature_schema_digest: str
    hash_algorithm: str

    true_label_id: int
    true_label_name: str

    prediction_id: int
    prediction_name: str

    confidence: float
    score_type: str
    topk_predictions: list

    features_digest: str
    top_features: list

    prev_hash: str
    record_hash: str


# Record creation
def make_multiclass_evidence(
    case_id,
    idx,
    context,
    model_name,
    true_label_id,
    true_label_name,
    pred_id,
    pred_name,
    confidence,
    topk_predictions,
    features_row,
    prev_hash,
    *,
    model_digest="",
    feature_schema_digest="",
    hash_algorithm="SHA-256",
    score_type="max_class_probability",
    top_features=None
):
    created = datetime.datetime.utcnow().isoformat() + "Z"
    event_time = context.get("timestamp") or context.get("time") or created

    if top_features is None:
        top_features = []

    features_digest = stable_hash(features_row)

    core = {
        "record_id": f"{case_id}-{idx}",
        "created_utc": created,
        "case_id": case_id,
        "event_time": event_time,
        "context": context,

        "model_name": model_name,
        "model_digest": model_digest,
        "feature_schema_digest": feature_schema_digest,
        "hash_algorithm": hash_algorithm,

        "true_label_id": int(true_label_id),
        "true_label_name": str(true_label_name),

        "prediction_id": int(pred_id),
        "prediction_name": str(pred_name),

        "confidence": float(confidence),
        "score_type": str(score_type),
        "topk_predictions": topk_predictions,

        "features_digest": features_digest,
        "top_features": top_features,

        "prev_hash": prev_hash
    }

    record_hash = stable_hash(core)
    return MultiClassEvidenceRecord(record_hash=record_hash, **core)

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")

def verify_jsonl_hashchain(path):
    prev = "GENESIS"
    ok = True
    bad_record = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            rh = rec["record_hash"]
            core = {k: rec[k] for k in rec.keys() if k != "record_hash"}

            if core["prev_hash"] != prev:
                print("Broken link at:", core.get("record_id"))
                ok = False
                bad_record = core.get("record_id")
                break

            if stable_hash(core) != rh:
                print("Hash mismatch at:", core.get("record_id"))
                ok = False
                bad_record = core.get("record_id")
                break

            prev = rh

    return ok, bad_record


# Utility: map transformed feature names back to raw feature names
def map_transformed_to_raw_feature_name(fname):
    if fname.endswith("__log1p"):
        return fname.replace("__log1p", "")
    if fname.endswith("__scaled"):
        return fname.replace("__scaled", "")
    if fname.endswith("__freq"):
        return fname.replace("__freq", "")

    for c in sorted(low_mid_card_cat_cols, key=len, reverse=True):
        if fname.startswith(c + "_"):
            return c
    return fname


# Prepare output paths
CASE_ID = f"CASE-{datetime.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
OUT_DIR = "forensic_outputs_multiclass"
os.makedirs(OUT_DIR, exist_ok=True)

EVIDENCE_LOG = os.path.join(OUT_DIR, f"{CASE_ID}_evidence_multiclass.jsonl")


# Model metadata digests
feature_schema_digest = stable_hash(list(X_test_raw.columns))

model_meta = {
    "model_name": best_name_final,
    "params": best_model.get_params(),
    "normal_class_id": int(NORMAL_CLASS_ID),
    "label_mapping": label_mapping,
    "hash_algorithm": HASH_ALGORITHM
}
model_digest = stable_hash(model_meta)


# Predictions and alert selection
t_pred0 = time.perf_counter()
y_pred = best_model.predict(X_test_raw)
y_proba = best_model.predict_proba(X_test_raw)
t_pred = time.perf_counter() - t_pred0

print("\n[Prediction runtime]")
print(f"Predict + predict_proba time (s): {t_pred:.4f}")

# alerts = non-normal predictions
alert_idxs = np.where(y_pred != NORMAL_CLASS_ID)[0]

print("\n[Alert summary]")
print("Total test samples        :", len(X_test_raw))
print("Predicted alerts          :", len(alert_idxs))
print("Predicted normal samples  :", int((y_pred == NORMAL_CLASS_ID).sum()))


# Rank alerts by confidence and keep top N for graph/log focus
N_ALERTS = min(500, len(alert_idxs))
alert_conf = y_proba[alert_idxs].max(axis=1)
top_alert_order = np.argsort(alert_conf)[::-1][:N_ALERTS]
top_alerts = alert_idxs[top_alert_order]

print("Alerts retained for evidence log:", len(top_alerts))


# Prepare SHAP values for the retained portion
# To keep runtime controlled, compute SHAP only for the retained alerts
TOPK_EXPLAIN = 5

X_test_r = X_test_raw.reset_index(drop=True)
ctx_test_r = ctx_test.reset_index(drop=True)
y_test_r = pd.Series(y_test).reset_index(drop=True)

X_alert_raw = X_test_r.iloc[top_alerts].copy()
X_alert_trans = preprocessor.transform(X_alert_raw)

if hasattr(X_alert_trans, "toarray"):
    X_alert_trans = X_alert_trans.toarray()
X_alert_trans = np.asarray(X_alert_trans)

print("\n[Per-alert SHAP setup]")
print("Retained alert sample size:", len(X_alert_raw))
print("Transformed alert matrix shape:", X_alert_trans.shape)

import shap
t_shap0 = time.perf_counter()
explainer_alert = shap.TreeExplainer(best_model.named_steps["clf"])
shap_values_alert = explainer_alert.shap_values(X_alert_trans)
t_shap_alert = time.perf_counter() - t_shap0

print(f"Per-alert SHAP computation time (s): {t_shap_alert:.4f}")

# normalize shape
if isinstance(shap_values_alert, list):
    shap_alert_by_class = [np.asarray(sv) for sv in shap_values_alert]
elif isinstance(shap_values_alert, np.ndarray) and shap_values_alert.ndim == 3:
    shap_alert_by_class = [shap_values_alert[:, :, k] for k in range(shap_values_alert.shape[2])]
else:
    raise ValueError(
        f"Unexpected SHAP output structure for retained alerts: "
        f"type={type(shap_values_alert)}, shape={getattr(shap_values_alert, 'shape', None)}"
    )


# Build and write evidence records
prev_hash = "GENESIS"

for j, i in enumerate(top_alerts):
    context = ctx_test_r.iloc[i].to_dict()
    feat_row = X_test_r.iloc[i].to_dict()

    pred_id = int(y_pred[i])
    pred_name = inv_label_mapping[pred_id]

    true_id = int(y_test_r.iloc[i])
    true_name = inv_label_mapping[true_id]

    proba_row = y_proba[i]
    confidence = float(np.max(proba_row))

    top3_idx = np.argsort(proba_row)[::-1][:3]
    topk_predictions = [
        {
            "class_id": int(k),
            "class_name": inv_label_mapping[int(k)],
            "probability": float(proba_row[k])
        }
        for k in top3_idx
    ]

    # per-alert SHAP for the predicted class
    local_alert_pos = j
    local_shap_vec = shap_alert_by_class[pred_id][local_alert_pos]

    local_shap_df = pd.DataFrame({
        "transformed_feature": transformed_feature_names,
        "shap_value": local_shap_vec
    })
    local_shap_df["abs_shap"] = local_shap_df["shap_value"].abs()
    local_shap_df["raw_feature"] = local_shap_df["transformed_feature"].apply(map_transformed_to_raw_feature_name)

    # aggregate to raw feature
    local_raw_shap = local_shap_df.groupby("raw_feature", as_index=False).agg({
        "shap_value": "sum",
        "abs_shap": "sum"
    }).sort_values(by="abs_shap", ascending=False).reset_index(drop=True)

    top_features = []
    for _, row in local_raw_shap.head(TOPK_EXPLAIN).iterrows():
        f = row["raw_feature"]
        val = X_test_r.loc[i, f] if f in X_test_r.columns else None

        if val is not None:
            try:
                val = float(val)
            except Exception:
                val = str(val)

        shap_val = float(row["shap_value"])
        sign = "+" if shap_val > 0 else "-" if shap_val < 0 else "0"

        top_features.append({
            "feature": f,
            "value": val,
            "shap_value": shap_val,
            "abs_shap_value": float(row["abs_shap"]),
            "sign": sign
        })

    rec = make_multiclass_evidence(
        case_id=CASE_ID,
        idx=j,
        context=context,
        model_name=best_name_final,
        true_label_id=true_id,
        true_label_name=true_name,
        pred_id=pred_id,
        pred_name=pred_name,
        confidence=confidence,
        topk_predictions=topk_predictions,
        features_row=feat_row,
        prev_hash=prev_hash,
        model_digest=model_digest,
        feature_schema_digest=feature_schema_digest,
        hash_algorithm=HASH_ALGORITHM,
        score_type="max_class_probability",
        top_features=top_features
    )

    append_jsonl(EVIDENCE_LOG, asdict(rec))
    prev_hash = rec.record_hash


# Final verification
hash_ok, bad_record = verify_jsonl_hashchain(EVIDENCE_LOG)
t_step6a = time.perf_counter() - t0

print("\n[Evidence log output]")
print("Case ID              :", CASE_ID)
print("Evidence log path    :", EVIDENCE_LOG)
print("Hash algorithm       :", HASH_ALGORITHM)
print("Hash-chain verified  :", hash_ok)
print("First bad record     :", bad_record)

# quick preview of first record
with open(EVIDENCE_LOG, "r", encoding="utf-8") as f:
    first_record_preview = json.loads(f.readline())

preview_keys = [
    "record_id", "prediction_name", "confidence",
    "topk_predictions", "top_features", "prev_hash", "record_hash", "hash_algorithm"
]
preview_obj = {k: first_record_preview.get(k) for k in preview_keys}

print("\n[First evidence record preview]")
print(json.dumps(preview_obj, indent=2))


# Save summary
step6a_summary = {
    "best_model": best_name_final,
    "case_id": CASE_ID,
    "evidence_log_path": EVIDENCE_LOG,
    "hash_algorithm": HASH_ALGORITHM,
    "test_samples_evaluated": int(len(X_test_raw)),
    "predicted_alerts": int(len(alert_idxs)),
    "predicted_normal_samples": int((y_pred == NORMAL_CLASS_ID).sum()),
    "alerts_retained_for_logging": int(len(top_alerts)),
    "prediction_runtime_seconds": float(t_pred),
    "per_alert_shap_runtime_seconds": float(t_shap_alert),
    "hash_chain_verified": bool(hash_ok),
    "first_bad_record": bad_record
}

with open(os.path.join(OUT_DIR, "step6a_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step6a_summary, f, indent=2)

print("\n[Saved summary]")
print(json.dumps(step6a_summary, indent=2))

print(f"\nSTEP 6A completed successfully in {t_step6a:.4f} seconds.")

In [ ]:
# STEP 6A2: Diversified alert selection for graph reconstruction
# - avoids graph collapse from a single dominant high-confidence class

import os
import json
import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 6A2: DIVERSIFIED ALERT SELECTION FOR GRAPH RECONSTRUCTION")
print("=" * 80)

required_vars = [
    "best_model", "X_test_raw", "y_test", "ctx_test",
    "inv_label_mapping", "label_mapping"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run earlier steps first."

if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

NORMAL_CLASS_ID = label_mapping["normal"]


# Predictions

X_test_r = X_test_raw.reset_index(drop=True)
ctx_test_r = ctx_test.reset_index(drop=True)
y_test_r = pd.Series(y_test).reset_index(drop=True)

y_pred = best_model.predict(X_test_r)
y_proba = best_model.predict_proba(X_test_r)

pred_conf = y_proba.max(axis=1)
pred_name = [inv_label_mapping[int(i)] for i in y_pred]

alerts_df = pd.DataFrame({
    "row_idx": np.arange(len(X_test_r)),
    "pred_id": y_pred,
    "pred_name": pred_name,
    "confidence": pred_conf,
    "true_id": y_test_r.values,
    "true_name": [inv_label_mapping[int(i)] for i in y_test_r.values]
})

alerts_df = pd.concat([alerts_df, ctx_test_r.reset_index(drop=True)], axis=1)
alerts_df = alerts_df[alerts_df["pred_id"] != NORMAL_CLASS_ID].copy().reset_index(drop=True)

print("\n[Initial malicious alert pool]")
print("Total non-normal predicted alerts:", len(alerts_df))
print("\nClass distribution before diversification:")
print(alerts_df["pred_name"].value_counts())


# Diversified selection
# 1) top-k per predicted class
# 2) top-k per host pair
# 3) merge and cap
PER_CLASS_CAP = 75
PER_PAIR_CAP = 10
FINAL_CAP = 500

selected_idx = set()

# top per class
for cname, grp in alerts_df.groupby("pred_name"):
    grp_sorted = grp.sort_values(by="confidence", ascending=False).head(PER_CLASS_CAP)
    selected_idx.update(grp_sorted["row_idx"].tolist())

# top per host pair
if "src_ip" in alerts_df.columns and "dst_ip" in alerts_df.columns:
    alerts_df["host_pair"] = alerts_df["src_ip"].astype(str) + "->" + alerts_df["dst_ip"].astype(str)
    for hp, grp in alerts_df.groupby("host_pair"):
        grp_sorted = grp.sort_values(by="confidence", ascending=False).head(PER_PAIR_CAP)
        selected_idx.update(grp_sorted["row_idx"].tolist())
else:
    alerts_df["host_pair"] = "UNKNOWN"

selected_df = alerts_df[alerts_df["row_idx"].isin(selected_idx)].copy()

# final cap by confidence if needed
selected_df = selected_df.sort_values(by="confidence", ascending=False).head(FINAL_CAP).reset_index(drop=True)

print("\n[Diversified graph-alert selection]")
print("Selected alerts for graph/log reconstruction:", len(selected_df))

print("\nClass distribution after diversification:")
print(selected_df["pred_name"].value_counts())

print("\nUnique hosts after diversification:")
host_set = set(selected_df["src_ip"].astype(str)).union(set(selected_df["dst_ip"].astype(str)))
print(len(host_set))

print("\nUnique host pairs after diversification:")
print(selected_df["host_pair"].nunique())


# Save diversified selection indices
OUT_DIR = "forensic_outputs_multiclass"
os.makedirs(OUT_DIR, exist_ok=True)

DIVERSIFIED_ALERT_INDEX_FILE = os.path.join(OUT_DIR, "diversified_graph_alert_selection.csv")
selected_df.to_csv(DIVERSIFIED_ALERT_INDEX_FILE, index=False)

diversified_summary = {
    "best_model": best_name_final,
    "per_class_cap": PER_CLASS_CAP,
    "per_pair_cap": PER_PAIR_CAP,
    "final_cap": FINAL_CAP,
    "selected_alert_count": int(len(selected_df)),
    "selected_class_distribution": selected_df["pred_name"].value_counts().to_dict(),
    "unique_hosts": int(len(host_set)),
    "unique_host_pairs": int(selected_df["host_pair"].nunique()),
    "selection_file": DIVERSIFIED_ALERT_INDEX_FILE
}

with open(os.path.join(OUT_DIR, "diversified_graph_alert_selection_summary.json"), "w", encoding="utf-8") as f:
    json.dump(diversified_summary, f, indent=2)

print("\n[Saved diversified selection artifacts]")
print(json.dumps(diversified_summary, indent=2))

print("\nSTEP 6A2 completed successfully.")

In [ ]:
# STEP 6A3: Diversified forensic evidence logging for graph reconstruction
# - writes a second evidence log from the diversified selection
# - includes per-alert signed SHAP attributions


import os
import json
import time
import hashlib
import datetime
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict

print("=" * 80)
print("STEP 6A3: DIVERSIFIED FORENSIC EVIDENCE LOGGING")
print("=" * 80)

t0 = time.perf_counter()

# Safety checks
required_vars = [
    "best_model",
    "X_test_raw", "y_test", "ctx_test",
    "inv_label_mapping", "label_mapping",
    "preprocessor", "transformed_feature_names",
    "low_mid_card_cat_cols",
    "DIVERSIFIED_ALERT_INDEX_FILE"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

assert os.path.exists(DIVERSIFIED_ALERT_INDEX_FILE), \
    f"Diversified selection file not found: {DIVERSIFIED_ALERT_INDEX_FILE}"

if "best_name_extended" in globals():
    best_name_final = best_name_extended
elif "scores_df_extended" in globals():
    best_name_final = scores_df_extended.loc[0, "model"]
else:
    best_name_final = "best_model"

assert "normal" in label_mapping, "'normal' class not found in label mapping."
NORMAL_CLASS_ID = label_mapping["normal"]


# Hash helpers
HASH_ALGORITHM = "SHA-256"

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def stable_hash(obj) -> str:
    s = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return sha256_hex(s)


# Evidence schema
@dataclass
class MultiClassEvidenceRecord:
    record_id: str
    created_utc: str
    case_id: str
    event_time: str
    context: dict

    model_name: str
    model_digest: str
    feature_schema_digest: str
    hash_algorithm: str

    true_label_id: int
    true_label_name: str

    prediction_id: int
    prediction_name: str

    confidence: float
    score_type: str
    topk_predictions: list

    features_digest: str
    top_features: list

    prev_hash: str
    record_hash: str

def make_multiclass_evidence(
    case_id,
    idx,
    context,
    model_name,
    true_label_id,
    true_label_name,
    pred_id,
    pred_name,
    confidence,
    topk_predictions,
    features_row,
    prev_hash,
    *,
    model_digest="",
    feature_schema_digest="",
    hash_algorithm="SHA-256",
    score_type="max_class_probability",
    top_features=None
):
    created = datetime.datetime.utcnow().isoformat() + "Z"
    event_time = context.get("timestamp") or context.get("time") or created

    if top_features is None:
        top_features = []

    features_digest = stable_hash(features_row)

    core = {
        "record_id": f"{case_id}-{idx}",
        "created_utc": created,
        "case_id": case_id,
        "event_time": event_time,
        "context": context,

        "model_name": model_name,
        "model_digest": model_digest,
        "feature_schema_digest": feature_schema_digest,
        "hash_algorithm": hash_algorithm,

        "true_label_id": int(true_label_id),
        "true_label_name": str(true_label_name),

        "prediction_id": int(pred_id),
        "prediction_name": str(pred_name),

        "confidence": float(confidence),
        "score_type": str(score_type),
        "topk_predictions": topk_predictions,

        "features_digest": features_digest,
        "top_features": top_features,

        "prev_hash": prev_hash
    }

    record_hash = stable_hash(core)
    return MultiClassEvidenceRecord(record_hash=record_hash, **core)

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")

def verify_jsonl_hashchain(path):
    prev = "GENESIS"
    ok = True
    bad_record = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            rh = rec["record_hash"]
            core = {k: rec[k] for k in rec.keys() if k != "record_hash"}

            if core["prev_hash"] != prev:
                ok = False
                bad_record = core.get("record_id")
                break

            if stable_hash(core) != rh:
                ok = False
                bad_record = core.get("record_id")
                break

            prev = rh

    return ok, bad_record


# Raw-feature mapper
def map_transformed_to_raw_feature_name(fname):
    if fname.endswith("__log1p"):
        return fname.replace("__log1p", "")
    if fname.endswith("__scaled"):
        return fname.replace("__scaled", "")
    if fname.endswith("__freq"):
        return fname.replace("__freq", "")

    for c in sorted(low_mid_card_cat_cols, key=len, reverse=True):
        if fname.startswith(c + "_"):
            return c
    return fname


# Load diversified selection
selected_df = pd.read_csv(DIVERSIFIED_ALERT_INDEX_FILE)
assert "row_idx" in selected_df.columns, "Expected 'row_idx' column not found."

selected_row_idxs = selected_df["row_idx"].astype(int).tolist()

print("\n[Diversified selection loaded]")
print("Selected alerts:", len(selected_row_idxs))
print("Class distribution:")
if "pred_name" in selected_df.columns:
    print(selected_df["pred_name"].value_counts())


# Prepare output paths
OUT_DIR = "forensic_outputs_multiclass"
os.makedirs(OUT_DIR, exist_ok=True)

CASE_ID_DIVERSIFIED = f"{CASE_ID}-DIVERSIFIED"
EVIDENCE_LOG_DIVERSIFIED = os.path.join(
    OUT_DIR,
    f"{CASE_ID_DIVERSIFIED}_evidence_multiclass.jsonl"
)

if os.path.exists(EVIDENCE_LOG_DIVERSIFIED):
    os.remove(EVIDENCE_LOG_DIVERSIFIED)


# Metadata digests
feature_schema_digest = stable_hash(list(X_test_raw.columns))

model_meta = {
    "model_name": best_name_final,
    "params": best_model.get_params(),
    "normal_class_id": int(NORMAL_CLASS_ID),
    "label_mapping": label_mapping,
    "hash_algorithm": HASH_ALGORITHM,
    "selection_policy": "diversified_graph_reconstruction"
}
model_digest = stable_hash(model_meta)


# Full test predictions
X_test_r = X_test_raw.reset_index(drop=True)
ctx_test_r = ctx_test.reset_index(drop=True)
y_test_r = pd.Series(y_test).reset_index(drop=True)

t_pred0 = time.perf_counter()
y_pred = best_model.predict(X_test_r)
y_proba = best_model.predict_proba(X_test_r)
t_pred = time.perf_counter() - t_pred0


# Compute SHAP only for selected alerts
X_sel_raw = X_test_r.iloc[selected_row_idxs].copy()
X_sel_trans = preprocessor.transform(X_sel_raw)

if hasattr(X_sel_trans, "toarray"):
    X_sel_trans = X_sel_trans.toarray()
X_sel_trans = np.asarray(X_sel_trans)

print("\n[Per-alert SHAP setup for diversified log]")
print("Selected raw rows:", len(X_sel_raw))
print("Transformed shape:", X_sel_trans.shape)

import shap
t_shap0 = time.perf_counter()
explainer_sel = shap.TreeExplainer(best_model.named_steps["clf"])
shap_values_sel = explainer_sel.shap_values(X_sel_trans)
t_shap = time.perf_counter() - t_shap0

print(f"Per-alert SHAP time (s): {t_shap:.4f}")

if isinstance(shap_values_sel, list):
    shap_sel_by_class = [np.asarray(sv) for sv in shap_values_sel]
elif isinstance(shap_values_sel, np.ndarray) and shap_values_sel.ndim == 3:
    shap_sel_by_class = [shap_values_sel[:, :, k] for k in range(shap_values_sel.shape[2])]
else:
    raise ValueError(
        f"Unexpected SHAP output structure: type={type(shap_values_sel)}, "
        f"shape={getattr(shap_values_sel, 'shape', None)}"
    )

# Write diversified evidence log
TOPK_EXPLAIN = 5
prev_hash = "GENESIS"

for j, i in enumerate(selected_row_idxs):
    context = ctx_test_r.iloc[i].to_dict()
    feat_row = X_test_r.iloc[i].to_dict()

    pred_id = int(y_pred[i])
    pred_name = inv_label_mapping[pred_id]

    true_id = int(y_test_r.iloc[i])
    true_name = inv_label_mapping[true_id]

    proba_row = y_proba[i]
    confidence = float(np.max(proba_row))

    top3_idx = np.argsort(proba_row)[::-1][:3]
    topk_predictions = [
        {
            "class_id": int(k),
            "class_name": inv_label_mapping[int(k)],
            "probability": float(proba_row[k])
        }
        for k in top3_idx
    ]

    local_shap_vec = shap_sel_by_class[pred_id][j]

    local_shap_df = pd.DataFrame({
        "transformed_feature": transformed_feature_names,
        "shap_value": local_shap_vec
    })
    local_shap_df["abs_shap"] = local_shap_df["shap_value"].abs()
    local_shap_df["raw_feature"] = local_shap_df["transformed_feature"].apply(map_transformed_to_raw_feature_name)

    local_raw_shap = local_shap_df.groupby("raw_feature", as_index=False).agg({
        "shap_value": "sum",
        "abs_shap": "sum"
    }).sort_values(by="abs_shap", ascending=False).reset_index(drop=True)

    top_features = []
    for _, row in local_raw_shap.head(TOPK_EXPLAIN).iterrows():
        f = row["raw_feature"]
        val = X_test_r.loc[i, f] if f in X_test_r.columns else None

        if val is not None:
            try:
                val = float(val)
            except Exception:
                val = str(val)

        shap_val = float(row["shap_value"])
        sign = "+" if shap_val > 0 else "-" if shap_val < 0 else "0"

        top_features.append({
            "feature": f,
            "value": val,
            "shap_value": shap_val,
            "abs_shap_value": float(row["abs_shap"]),
            "sign": sign
        })

    rec = make_multiclass_evidence(
        case_id=CASE_ID_DIVERSIFIED,
        idx=j,
        context=context,
        model_name=best_name_final,
        true_label_id=true_id,
        true_label_name=true_name,
        pred_id=pred_id,
        pred_name=pred_name,
        confidence=confidence,
        topk_predictions=topk_predictions,
        features_row=feat_row,
        prev_hash=prev_hash,
        model_digest=model_digest,
        feature_schema_digest=feature_schema_digest,
        hash_algorithm=HASH_ALGORITHM,
        score_type="max_class_probability",
        top_features=top_features
    )

    append_jsonl(EVIDENCE_LOG_DIVERSIFIED, asdict(rec))
    prev_hash = rec.record_hash

hash_ok_div, bad_record_div = verify_jsonl_hashchain(EVIDENCE_LOG_DIVERSIFIED)

print("\n[Diversified evidence log output]")
print("Case ID                  :", CASE_ID_DIVERSIFIED)
print("Evidence log path        :", EVIDENCE_LOG_DIVERSIFIED)
print("Hash algorithm           :", HASH_ALGORITHM)
print("Hash-chain verified      :", hash_ok_div)
print("First bad record         :", bad_record_div)

# preview
with open(EVIDENCE_LOG_DIVERSIFIED, "r", encoding="utf-8") as f:
    preview_div = json.loads(f.readline())

preview_keys = [
    "record_id", "prediction_name", "confidence",
    "topk_predictions", "top_features", "prev_hash", "record_hash", "hash_algorithm"
]
preview_obj_div = {k: preview_div.get(k) for k in preview_keys}

print("\n[First diversified evidence record preview]")
print(json.dumps(preview_obj_div, indent=2))

step6a3_summary = {
    "best_model": best_name_final,
    "case_id_diversified": CASE_ID_DIVERSIFIED,
    "evidence_log_diversified_path": EVIDENCE_LOG_DIVERSIFIED,
    "hash_algorithm": HASH_ALGORITHM,
    "selected_alerts_for_diversified_log": int(len(selected_row_idxs)),
    "prediction_runtime_seconds": float(t_pred),
    "per_alert_shap_runtime_seconds": float(t_shap),
    "hash_chain_verified": bool(hash_ok_div),
    "first_bad_record": bad_record_div
}

with open(os.path.join(OUT_DIR, "step6a3_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step6a3_summary, f, indent=2)

print("\n[Saved summary]")
print(json.dumps(step6a3_summary, indent=2))

print(f"\nSTEP 6A3 completed successfully in {time.perf_counter() - t0:.4f} seconds.")

In [ ]:
# STEP 6B / STEP 7: Multiclass evidence graph construction
# + host interaction graph + graph statistics

import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter, defaultdict

print("=" * 80)
print("STEP 6B / STEP 7: MULTICLASS EVIDENCE GRAPH + HOST GRAPH")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks

required_vars = [
    "CASE_ID", "inv_label_mapping"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

GRAPH_EVIDENCE_LOG = EVIDENCE_LOG_DIVERSIFIED if "EVIDENCE_LOG_DIVERSIFIED" in globals() else EVIDENCE_LOG
assert os.path.exists(GRAPH_EVIDENCE_LOG), f"Evidence log not found: {GRAPH_EVIDENCE_LOG}"


# Graph containers
G = nx.DiGraph()   # evidence graph: hosts + evidence nodes
H = nx.DiGraph()   # host interaction graph

host_pair_counts = Counter()
host_pair_class_counts = defaultdict(Counter)
host_alert_counts = Counter()
class_counts = Counter()
confidence_by_class = defaultdict(list)


# Helper functions
def add_node_safe(graph, node_id, **attrs):
    if node_id is None or node_id == "":
        return
    if not graph.has_node(node_id):
        graph.add_node(node_id, **attrs)
    else:
        for k, v in attrs.items():
            if k not in graph.nodes[node_id]:
                graph.nodes[node_id][k] = v

def add_edge_weighted(graph, u, v, relation=None, **attrs):
    if not u or not v:
        return

    if graph.has_edge(u, v):
        graph[u][v]["weight"] = graph[u][v].get("weight", 1) + 1
    else:
        graph.add_edge(u, v, relation=relation, weight=1, **attrs)


# Read evidence log and construct graphs
with open(GRAPH_EVIDENCE_LOG, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)

        rid = rec["record_id"]
        pred_name = rec.get("prediction_name", "unknown")
        pred_id = rec.get("prediction_id", -1)
        true_name = rec.get("true_label_name", "unknown")
        confidence = float(rec.get("confidence", 0.0))
        etime = rec.get("event_time", "")
        context = rec.get("context", {}) if isinstance(rec.get("context"), dict) else {}

        src_ip = context.get("src_ip")
        dst_ip = context.get("dst_ip")
        src_port = context.get("src_port")
        dst_port = context.get("dst_port")
        proto = context.get("proto")
        service = context.get("service")
        conn_state = context.get("conn_state")

       
        # Evidence node  
        add_node_safe(
            G,
            rid,
            kind="evidence",
            prediction_name=pred_name,
            prediction_id=pred_id,
            true_label_name=true_name,
            confidence=confidence,
            event_time=etime
        )

        class_counts[pred_name] += 1
        confidence_by_class[pred_name].append(confidence)

        
        # Host nodes + evidence edges        
        if src_ip:
            add_node_safe(G, src_ip, kind="host")
            add_edge_weighted(
                G, src_ip, rid,
                relation="generated",
                proto=proto,
                service=service,
                conn_state=conn_state
            )
            host_alert_counts[src_ip] += 1

        if dst_ip:
            add_node_safe(G, dst_ip, kind="host")
            add_edge_weighted(
                G, rid, dst_ip,
                relation="targeted",
                proto=proto,
                service=service,
                conn_state=conn_state
            )
            host_alert_counts[dst_ip] += 1

        
        # Host-to-host graph        
        if src_ip and dst_ip:
            host_pair_counts[(src_ip, dst_ip)] += 1
            host_pair_class_counts[(src_ip, dst_ip)][pred_name] += 1

            if not H.has_node(src_ip):
                H.add_node(src_ip, kind="host")
            if not H.has_node(dst_ip):
                H.add_node(dst_ip, kind="host")

            if H.has_edge(src_ip, dst_ip):
                H[src_ip][dst_ip]["weight"] += 1
                H[src_ip][dst_ip]["class_counts"][pred_name] = H[src_ip][dst_ip]["class_counts"].get(pred_name, 0) + 1
            else:
                H.add_edge(
                    src_ip, dst_ip,
                    relation="communicates",
                    weight=1,
                    class_counts={pred_name: 1}
                )


# Graph summary statistics
t_graph = time.perf_counter() - t0

print("\n[Evidence graph summary]")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\n[Host interaction graph summary]")
print("Nodes:", H.number_of_nodes())
print("Edges:", H.number_of_edges())

print(f"\nGraph construction time (s): {t_graph:.4f}")


# Degree centrality
evidence_graph_centrality = nx.degree_centrality(G)
host_graph_centrality = nx.degree_centrality(H)

evidence_centrality_df = pd.DataFrame({
    "node": list(evidence_graph_centrality.keys()),
    "degree_centrality": list(evidence_graph_centrality.values())
}).sort_values(by="degree_centrality", ascending=False).reset_index(drop=True)

host_centrality_df = pd.DataFrame({
    "host": list(host_graph_centrality.keys()),
    "degree_centrality": list(host_graph_centrality.values())
}).sort_values(by="degree_centrality", ascending=False).reset_index(drop=True)

print("\n[Top evidence-graph nodes by degree centrality]")
print(evidence_centrality_df.head(15))

print("\n[Top host-graph nodes by degree centrality]")
print(host_centrality_df.head(15))



# Top nodes by degree
top_evidence_graph_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:15]
top_host_graph_nodes = sorted(H.degree, key=lambda x: x[1], reverse=True)[:15]

print("\n[Top evidence-graph nodes by degree]")
print(top_evidence_graph_nodes)

print("\n[Top host-graph nodes by degree]")
print(top_host_graph_nodes)


# Alert class distribution in evidence graph
class_dist_df = pd.DataFrame({
    "predicted_class": list(class_counts.keys()),
    "count": list(class_counts.values()),
    "percent": [100 * c / max(sum(class_counts.values()), 1) for c in class_counts.values()],
    "mean_confidence": [np.mean(confidence_by_class[k]) if len(confidence_by_class[k]) > 0 else 0.0 for k in class_counts.keys()]
}).sort_values(by="count", ascending=False).reset_index(drop=True)

print("\n[Evidence-node class distribution]")
print(class_dist_df)


# Top suspicious host pairs
pair_rows = []
for (src, dst), w in host_pair_counts.items():
    class_counter = host_pair_class_counts[(src, dst)]
    top_class, top_class_count = class_counter.most_common(1)[0]

    pair_rows.append({
        "src_ip": src,
        "dst_ip": dst,
        "edge_weight": w,
        "dominant_class": top_class,
        "dominant_class_count": top_class_count,
        "class_breakdown": dict(class_counter)
    })

host_pair_df = pd.DataFrame(pair_rows).sort_values(
    by=["edge_weight", "dominant_class_count"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n[Top 20 suspicious host pairs]")
print(host_pair_df.head(20))


# Top suspicious hosts
host_alert_df = pd.DataFrame({
    "host": list(host_alert_counts.keys()),
    "alert_count": list(host_alert_counts.values())
}).sort_values(by="alert_count", ascending=False).reset_index(drop=True)

print("\n[Top 20 suspicious hosts by alert participation]")
print(host_alert_df.head(20))


# Edge-level class summary for host graph
host_graph_edge_rows = []
for u, v, data in H.edges(data=True):
    class_counter = data.get("class_counts", {})
    dominant_class = None
    dominant_count = 0
    if len(class_counter) > 0:
        dominant_class, dominant_count = sorted(class_counter.items(), key=lambda x: x[1], reverse=True)[0]

    host_graph_edge_rows.append({
        "src_ip": u,
        "dst_ip": v,
        "weight": data.get("weight", 0),
        "dominant_class": dominant_class,
        "dominant_class_count": dominant_count,
        "class_counts": class_counter
    })

host_graph_edges_df = pd.DataFrame(host_graph_edge_rows).sort_values(
    by=["weight", "dominant_class_count"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n[Top 20 host-graph edges]")
print(host_graph_edges_df.head(20))


# Save graph artifacts
OUT_DIR = "multiclass_graph_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class_dist_df.to_csv(os.path.join(OUT_DIR, "evidence_class_distribution.csv"), index=False)
host_pair_df.to_csv(os.path.join(OUT_DIR, "top_host_pairs.csv"), index=False)
host_alert_df.to_csv(os.path.join(OUT_DIR, "top_hosts_by_alert_count.csv"), index=False)
host_graph_edges_df.to_csv(os.path.join(OUT_DIR, "host_graph_edges.csv"), index=False)


# Make GraphML-safe copies of graphs
# GraphML does not support dict/list attributes directly
def make_graphml_safe_graph(graph):
    G_safe = graph.copy()

    # node attributes
    for n, attrs in G_safe.nodes(data=True):
        for k, v in list(attrs.items()):
            if isinstance(v, (dict, list, tuple, set)):
                G_safe.nodes[n][k] = json.dumps(v, ensure_ascii=False, default=str)
            elif v is None:
                G_safe.nodes[n][k] = ""
            elif isinstance(v, bool):
                G_safe.nodes[n][k] = int(v)

    # edge attributes
    for u, v, attrs in G_safe.edges(data=True):
        for k, val in list(attrs.items()):
            if isinstance(val, (dict, list, tuple, set)):
                G_safe[u][v][k] = json.dumps(val, ensure_ascii=False, default=str)
            elif val is None:
                G_safe[u][v][k] = ""
            elif isinstance(val, bool):
                G_safe[u][v][k] = int(val)

    return G_safe

G_graphml = make_graphml_safe_graph(G)
H_graphml = make_graphml_safe_graph(H)

nx.write_graphml(G_graphml, os.path.join(OUT_DIR, f"{CASE_ID}_evidence_graph.graphml"))
nx.write_graphml(H_graphml, os.path.join(OUT_DIR, f"{CASE_ID}_host_graph.graphml"))

graph_summary = {
    "case_id": CASE_ID,
    "graph_evidence_log": GRAPH_EVIDENCE_LOG,
    "evidence_graph_nodes": int(G.number_of_nodes()),
    "evidence_graph_edges": int(G.number_of_edges()),
    "host_graph_nodes": int(H.number_of_nodes()),
    "host_graph_edges": int(H.number_of_edges()),
    "predicted_alert_count_logged": int(sum(class_counts.values())),
    "class_distribution": class_dist_df.to_dict(orient="records")[:20]
}

with open(os.path.join(OUT_DIR, "graph_summary.json"), "w", encoding="utf-8") as f:
    json.dump(graph_summary, f, indent=2)

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Graph summary]")
print(json.dumps(graph_summary, indent=2))

print(f"\nSTEP 6B / STEP 7 completed successfully in {t_graph:.4f} seconds.")


# Top nodes by degree
top_evidence_graph_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:15]
top_host_graph_nodes = sorted(H.degree, key=lambda x: x[1], reverse=True)[:15]

print("\n[Top evidence-graph nodes by degree]")
print(top_evidence_graph_nodes)

print("\n[Top host-graph nodes by degree]")
print(top_host_graph_nodes)


# Alert class distribution in evidence graph
class_dist_df = pd.DataFrame({
    "predicted_class": list(class_counts.keys()),
    "count": list(class_counts.values()),
    "percent": [100 * c / max(sum(class_counts.values()), 1) for c in class_counts.values()],
    "mean_confidence": [np.mean(confidence_by_class[k]) if len(confidence_by_class[k]) > 0 else 0.0 for k in class_counts.keys()]
}).sort_values(by="count", ascending=False).reset_index(drop=True)

print("\n[Evidence-node class distribution]")
print(class_dist_df)


# Top suspicious host pairs
pair_rows = []
for (src, dst), w in host_pair_counts.items():
    class_counter = host_pair_class_counts[(src, dst)]
    top_class, top_class_count = class_counter.most_common(1)[0]

    pair_rows.append({
        "src_ip": src,
        "dst_ip": dst,
        "edge_weight": w,
        "dominant_class": top_class,
        "dominant_class_count": top_class_count,
        "class_breakdown": dict(class_counter)
    })

host_pair_df = pd.DataFrame(pair_rows).sort_values(
    by=["edge_weight", "dominant_class_count"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n[Top 20 suspicious host pairs]")
print(host_pair_df.head(20))


# Top suspicious hosts
host_alert_df = pd.DataFrame({
    "host": list(host_alert_counts.keys()),
    "alert_count": list(host_alert_counts.values())
}).sort_values(by="alert_count", ascending=False).reset_index(drop=True)

print("\n[Top 20 suspicious hosts by alert participation]")
print(host_alert_df.head(20))


# Edge-level class summary for host graph
host_graph_edge_rows = []
for u, v, data in H.edges(data=True):
    class_counter = data.get("class_counts", {})
    dominant_class = None
    dominant_count = 0
    if len(class_counter) > 0:
        dominant_class, dominant_count = sorted(class_counter.items(), key=lambda x: x[1], reverse=True)[0]

    host_graph_edge_rows.append({
        "src_ip": u,
        "dst_ip": v,
        "weight": data.get("weight", 0),
        "dominant_class": dominant_class,
        "dominant_class_count": dominant_count,
        "class_counts": class_counter
    })

host_graph_edges_df = pd.DataFrame(host_graph_edge_rows).sort_values(
    by=["weight", "dominant_class_count"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n[Top 20 host-graph edges]")
print(host_graph_edges_df.head(20))


# Save graph artifacts
OUT_DIR = "multiclass_graph_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class_dist_df.to_csv(os.path.join(OUT_DIR, "evidence_class_distribution.csv"), index=False)
host_pair_df.to_csv(os.path.join(OUT_DIR, "top_host_pairs.csv"), index=False)
host_alert_df.to_csv(os.path.join(OUT_DIR, "top_hosts_by_alert_count.csv"), index=False)
host_graph_edges_df.to_csv(os.path.join(OUT_DIR, "host_graph_edges.csv"), index=False)
evidence_centrality_df.to_csv(os.path.join(OUT_DIR, "evidence_graph_degree_centrality.csv"), index=False)
host_centrality_df.to_csv(os.path.join(OUT_DIR, "host_graph_degree_centrality.csv"), index=False)

# Make GraphML-safe copies of graphs
# GraphML does not support dict/list attributes directly
def make_graphml_safe_graph(graph):
    G_safe = graph.copy()

    # node attributes
    for n, attrs in G_safe.nodes(data=True):
        for k, v in list(attrs.items()):
            if isinstance(v, (dict, list, tuple, set)):
                G_safe.nodes[n][k] = json.dumps(v, ensure_ascii=False, default=str)
            elif v is None:
                G_safe.nodes[n][k] = ""
            elif isinstance(v, bool):
                G_safe.nodes[n][k] = int(v)

    # edge attributes
    for u, v, attrs in G_safe.edges(data=True):
        for k, val in list(attrs.items()):
            if isinstance(val, (dict, list, tuple, set)):
                G_safe[u][v][k] = json.dumps(val, ensure_ascii=False, default=str)
            elif val is None:
                G_safe[u][v][k] = ""
            elif isinstance(val, bool):
                G_safe[u][v][k] = int(val)

    return G_safe

G_graphml = make_graphml_safe_graph(G)
H_graphml = make_graphml_safe_graph(H)

nx.write_graphml(G_graphml, os.path.join(OUT_DIR, f"{CASE_ID}_evidence_graph.graphml"))
nx.write_graphml(H_graphml, os.path.join(OUT_DIR, f"{CASE_ID}_host_graph.graphml"))

graph_summary = {
    "case_id": CASE_ID,
    "evidence_graph_nodes": int(G.number_of_nodes()),
    "evidence_graph_edges": int(G.number_of_edges()),
    "host_graph_nodes": int(H.number_of_nodes()),
    "host_graph_edges": int(H.number_of_edges()),
    "predicted_alert_count_logged": int(sum(class_counts.values())),
    "class_distribution": class_dist_df.to_dict(orient="records")[:20]
}

with open(os.path.join(OUT_DIR, "graph_summary.json"), "w", encoding="utf-8") as f:
    json.dump(graph_summary, f, indent=2)

print("\n[Saved artifacts]")
print("Directory:", OUT_DIR)

print("\n[Graph summary]")
print(json.dumps(graph_summary, indent=2))

print(f"\nSTEP 6B / STEP 7 completed successfully in {t_graph:.4f} seconds.")

In [ ]:
# STEP 7B:graph visualization
# - evidence graph
# - host interaction graph
# - class-aware host graph


import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

print("=" * 80)
print("STEP 7B: GRAPH VISUALIZATION")
print("=" * 80)


# Safety checks
required_vars = ["G", "H", "CASE_ID"]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run Step 6B/7 first."

OUT_DIR = "multiclass_graph_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# Class-to-color mapping (stable, readable)
# We let matplotlib choose named colors for clarity
CLASS_COLOR_MAP = {
    "backdoor": "tab:red",
    "ddos": "tab:orange",
    "dos": "tab:brown",
    "injection": "tab:purple",
    "normal": "tab:green",
    "password": "tab:pink",
    "ransomware": "tab:gray",
    "scanning": "tab:blue",
    "xss": "tab:olive",
    "unknown": "black"
}

GRAPH_CASE_PREFIX = CASE_ID_DIVERSIFIED if "CASE_ID_DIVERSIFIED" in globals() else CASE_ID


# 1) Evidence graph: curated top-degree subgraph
def plot_evidence_graph_topk(G, k=60, seed=42, save_prefix="evidence_graph_topk"):
    top_nodes = [n for n, d in sorted(G.degree, key=lambda x: x[1], reverse=True)[:k]]
    SG = G.subgraph(top_nodes).copy()

    plt.figure(figsize=(12, 9), dpi=180)
    pos = nx.spring_layout(SG, seed=seed, k=0.9)

    host_nodes = [n for n in SG.nodes() if SG.nodes[n].get("kind") == "host"]
    evidence_nodes = [n for n in SG.nodes() if SG.nodes[n].get("kind") == "evidence"]

    host_sizes = [350 + 25 * SG.degree(n) for n in host_nodes]
    evidence_sizes = [120 + 10 * SG.degree(n) for n in evidence_nodes]

    # evidence node color by predicted class
    evidence_colors = [
        CLASS_COLOR_MAP.get(SG.nodes[n].get("prediction_name", "unknown"), "black")
        for n in evidence_nodes
    ]

    nx.draw_networkx_edges(
        SG, pos,
        alpha=0.18,
        arrows=True,
        arrowsize=8,
        width=0.8
    )

    nx.draw_networkx_nodes(
        SG, pos,
        nodelist=host_nodes,
        node_size=host_sizes,
        node_color="skyblue",
        alpha=0.95,
        edgecolors="black",
        linewidths=0.5
    )

    nx.draw_networkx_nodes(
        SG, pos,
        nodelist=evidence_nodes,
        node_size=evidence_sizes,
        node_color=evidence_colors,
        alpha=0.65,
        edgecolors="none"
    )

    label_nodes = [n for n, d in sorted(SG.degree, key=lambda x: x[1], reverse=True)[:12]]
    labels = {n: str(n) for n in label_nodes}
    nx.draw_networkx_labels(SG, pos, labels=labels, font_size=8)

    plt.title("Evidence graph (hosts + multiclass evidence nodes, top-degree subgraph)")
    plt.axis("off")
    plt.tight_layout()

    png_path = os.path.join(OUT_DIR, f"{save_prefix}.png")
    pdf_path = os.path.join(OUT_DIR, f"{save_prefix}.pdf")
    plt.savefig(png_path, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return png_path, pdf_path


# 2) Host interaction graph: top weighted edges
def plot_host_graph_top_edges(H, top_edges=30, seed=42, save_prefix="host_graph_top_edges"):
    edges_sorted = sorted(H.edges(data=True), key=lambda e: e[2].get("weight", 1), reverse=True)[:top_edges]

    keep_nodes = set()
    for u, v, _ in edges_sorted:
        keep_nodes.add(u)
        keep_nodes.add(v)

    SG = H.subgraph(list(keep_nodes)).copy()

    plt.figure(figsize=(11, 8), dpi=180)
    pos = nx.spring_layout(SG, seed=seed, k=1.0)

    node_sizes = [500 + 100 * SG.degree(n) for n in SG.nodes()]
    edge_widths = [0.8 + 0.15 * SG[u][v].get("weight", 1) for u, v in SG.edges()]

    nx.draw_networkx_nodes(
        SG, pos,
        node_size=node_sizes,
        node_color="lightsteelblue",
        alpha=0.95,
        edgecolors="black",
        linewidths=0.5
    )
    nx.draw_networkx_edges(
        SG, pos,
        width=edge_widths,
        alpha=0.35,
        arrows=True,
        arrowsize=10
    )

    # labels = {n: str(n) for n in sorted(SG.degree, key=lambda x: x[1], reverse=True)[:12]}
    top_label_nodes = [n for n, d in sorted(SG.degree, key=lambda x: x[1], reverse=True)[:12]]
    labels = {n: str(n) for n in top_label_nodes}
    nx.draw_networkx_labels(SG, pos, labels=labels, font_size=8)

    plt.title("Host interaction graph (top weighted suspicious edges)")
    plt.axis("off")
    plt.tight_layout()

    png_path = os.path.join(OUT_DIR, f"{save_prefix}.png")
    pdf_path = os.path.join(OUT_DIR, f"{save_prefix}.pdf")
    plt.savefig(png_path, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return png_path, pdf_path


# 3) Class-aware host graph
# Edge color by dominant predicted class
def plot_class_aware_host_graph(H, top_edges=30, seed=42, save_prefix="host_graph_class_aware"):
    edges_sorted = sorted(H.edges(data=True), key=lambda e: e[2].get("weight", 1), reverse=True)[:top_edges]

    keep_nodes = set()
    for u, v, _ in edges_sorted:
        keep_nodes.add(u)
        keep_nodes.add(v)

    SG = H.subgraph(list(keep_nodes)).copy()

    plt.figure(figsize=(11, 8), dpi=180)
    pos = nx.spring_layout(SG, seed=seed, k=1.0)

    node_sizes = [500 + 100 * SG.degree(n) for n in SG.nodes()]

    edge_colors = []
    edge_widths = []

    for u, v, data in SG.edges(data=True):
        class_counts = data.get("class_counts", {})
        dominant_class = "unknown"
        if isinstance(class_counts, dict) and len(class_counts) > 0:
            dominant_class = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[0][0]

        edge_colors.append(CLASS_COLOR_MAP.get(dominant_class, "black"))
        edge_widths.append(0.8 + 0.15 * data.get("weight", 1))

    nx.draw_networkx_nodes(
        SG, pos,
        node_size=node_sizes,
        node_color="whitesmoke",
        alpha=0.98,
        edgecolors="black",
        linewidths=0.7
    )
    nx.draw_networkx_edges(
        SG, pos,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.55,
        arrows=True,
        arrowsize=10
    )

    # labels = {n: str(n) for n in sorted(SG.degree, key=lambda x: x[1], reverse=True)[:12]}
    top_label_nodes = [n for n, d in sorted(SG.degree, key=lambda x: x[1], reverse=True)[:12]]
    labels = {n: str(n) for n in top_label_nodes}
    nx.draw_networkx_labels(SG, pos, labels=labels, font_size=8)

    # legend
    legend_classes = sorted({ 
        sorted(data.get("class_counts", {}).items(), key=lambda x: x[1], reverse=True)[0][0]
        for _, _, data in SG.edges(data=True)
        if isinstance(data.get("class_counts", {}), dict) and len(data.get("class_counts", {})) > 0
    })

    handles = []
    for cls in legend_classes:
        handles.append(plt.Line2D([0], [0], color=CLASS_COLOR_MAP.get(cls, "black"), lw=3, label=cls))

    if handles:
        plt.legend(handles=handles, title="Dominant predicted class", loc="best", fontsize=8)

    plt.title("Class-aware host interaction graph (edge color = dominant class)")
    plt.axis("off")
    plt.tight_layout()

    png_path = os.path.join(OUT_DIR, f"{save_prefix}.png")
    pdf_path = os.path.join(OUT_DIR, f"{save_prefix}.pdf")
    plt.savefig(png_path, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return png_path, pdf_path

ev_png, ev_pdf = plot_evidence_graph_topk(G, k=60, seed=42, save_prefix=f"{GRAPH_CASE_PREFIX}_evidence_graph_topk")
hg_png, hg_pdf = plot_host_graph_top_edges(H, top_edges=30, seed=42, save_prefix=f"{GRAPH_CASE_PREFIX}_host_graph_top_edges")
cg_png, cg_pdf = plot_class_aware_host_graph(H, top_edges=30, seed=42, save_prefix=f"{GRAPH_CASE_PREFIX}_host_graph_class_aware")

print("\n[Saved plot files]")
print(ev_png)
print(ev_pdf)
print(hg_png)
print(hg_pdf)
print(cg_png)
print(cg_pdf)

print("\nSTEP 7B completed successfully.")

In [ ]:
# STEP 8: FORENSIC TIMELINE RECONSTRUCTION
# - sequence-based timeline from evidence log order / created_utc
# - class activity over sequence
# - cumulative alert evolution
# - class transition analysis


import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter

print("=" * 80)
print("STEP 8: FORENSIC TIMELINE RECONSTRUCTION")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = ["CASE_ID"]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

TIMELINE_EVIDENCE_LOG = EVIDENCE_LOG_DIVERSIFIED if "EVIDENCE_LOG_DIVERSIFIED" in globals() else EVIDENCE_LOG
assert os.path.exists(TIMELINE_EVIDENCE_LOG), f"Evidence log not found: {TIMELINE_EVIDENCE_LOG}"

TIMELINE_CASE_ID = CASE_ID_DIVERSIFIED if "CASE_ID_DIVERSIFIED" in globals() else CASE_ID

OUT_DIR = "multiclass_timeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


# Load evidence log
records = []
with open(TIMELINE_EVIDENCE_LOG, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        records.append(rec)

assert len(records) > 0, "Evidence log is empty."

timeline_df = pd.DataFrame(records)

print("\n[Evidence log loaded]")
print("Evidence source:", TIMELINE_EVIDENCE_LOG)
print("Number of logged alerts:", len(timeline_df))
print("Columns:")
print(timeline_df.columns.tolist())


# Normalize timeline fields
timeline_df["created_utc"] = pd.to_datetime(timeline_df["created_utc"], errors="coerce", utc=True)
timeline_df["event_time"] = pd.to_datetime(timeline_df["event_time"], errors="coerce", utc=True)

# Sequence-based ordering
timeline_df = timeline_df.sort_values(by=["created_utc", "record_id"]).reset_index(drop=True)
timeline_df["sequence_id"] = np.arange(1, len(timeline_df) + 1)

valid_event_time_ratio = timeline_df["event_time"].notna().mean()

print("\n[Timeline parsing status]")
print(f"Valid created_utc ratio : {timeline_df['created_utc'].notna().mean():.4f}")
print(f"Valid event_time ratio  : {valid_event_time_ratio:.4f}")


# Basic timeline summaries
timeline_summary = {
    "case_id": TIMELINE_CASE_ID,
    "timeline_evidence_log": TIMELINE_EVIDENCE_LOG,
    "logged_alerts": int(len(timeline_df)),
    "classes_present": sorted(timeline_df["prediction_name"].dropna().unique().tolist()),
    "timeline_start_created_utc": str(timeline_df["created_utc"].min()),
    "timeline_end_created_utc": str(timeline_df["created_utc"].max()),
    "mean_confidence": float(pd.to_numeric(timeline_df["confidence"], errors="coerce").mean()),
    "timeline_type": "sequence_based_forensic_timeline"
}

print("\n[Timeline summary]")
print(json.dumps(timeline_summary, indent=2))


# Class distribution over case sequence
class_counts = timeline_df["prediction_name"].value_counts().sort_values(ascending=False)
print("\n[Class counts in forensic timeline]")
print(class_counts)


# Bin sequence into windows
N_WINDOWS = min(20, max(5, len(timeline_df) // 25))
timeline_df["window_id"] = pd.cut(
    timeline_df["sequence_id"],
    bins=N_WINDOWS,
    labels=False,
    include_lowest=True
)

window_class_counts = (
    timeline_df.groupby(["window_id", "prediction_name"])
    .size()
    .reset_index(name="count")
)

window_pivot = window_class_counts.pivot(
    index="window_id", columns="prediction_name", values="count"
).fillna(0).astype(int)

print("\n[Windowed class activity table]")
print(window_pivot)


# Plot 1: class activity over sequence windows
plt.figure(figsize=(12, 6), dpi=180)
for cname in window_pivot.columns:
    plt.plot(window_pivot.index, window_pivot[cname], marker="o", label=cname)

plt.title("Class activity across forensic sequence windows")
plt.xlabel("Sequence window")
plt.ylabel("Alert count")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
timeline_plot1_png = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_class_activity.png")
timeline_plot1_pdf = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_class_activity.pdf")
plt.savefig(timeline_plot1_png, bbox_inches="tight")
plt.savefig(timeline_plot1_pdf, bbox_inches="tight")
plt.show()


# Plot 2: stacked class activity
window_pivot_plot = window_pivot.copy()

plt.figure(figsize=(12, 6), dpi=180)
bottom = np.zeros(len(window_pivot_plot))

for cname in window_pivot_plot.columns:
    vals = window_pivot_plot[cname].values
    plt.bar(window_pivot_plot.index, vals, bottom=bottom, label=cname)
    bottom += vals

plt.title("Stacked forensic alert sequence by predicted class")
plt.xlabel("Sequence window")
plt.ylabel("Alert count")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
timeline_plot2_png = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_stacked_class_activity.png")
timeline_plot2_pdf = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_stacked_class_activity.pdf")
plt.savefig(timeline_plot2_png, bbox_inches="tight")
plt.savefig(timeline_plot2_pdf, bbox_inches="tight")
plt.show()


# Plot 3: cumulative alert growth by class
cum_df = pd.DataFrame({"sequence_id": timeline_df["sequence_id"]})

for cname in sorted(timeline_df["prediction_name"].dropna().unique()):
    cum_df[cname] = (timeline_df["prediction_name"] == cname).astype(int).cumsum()

plt.figure(figsize=(12, 6), dpi=180)
for cname in cum_df.columns[1:]:
    plt.plot(cum_df["sequence_id"], cum_df[cname], label=cname)

plt.title("Cumulative multiclass alert growth over forensic sequence")
plt.xlabel("Alert sequence")
plt.ylabel("Cumulative count")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
timeline_plot3_png = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_cumulative_alerts.png")
timeline_plot3_pdf = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_cumulative_alerts.pdf")
plt.savefig(timeline_plot3_png, bbox_inches="tight")
plt.savefig(timeline_plot3_pdf, bbox_inches="tight")
plt.show()


# Transition analysis between consecutive predicted classes
transitions = []
pred_seq = timeline_df["prediction_name"].tolist()

for i in range(len(pred_seq) - 1):
    transitions.append((pred_seq[i], pred_seq[i + 1]))

transition_counts = Counter(transitions)

transition_df = pd.DataFrame([
    {"from_class": a, "to_class": b, "count": c}
    for (a, b), c in transition_counts.items()
]).sort_values(by="count", ascending=False).reset_index(drop=True)

print("\n[Top class transitions]")
print(transition_df.head(20))


# Transition graph
TOP_TRANSITIONS = min(20, len(transition_df))
top_transition_edges = transition_df.head(TOP_TRANSITIONS)

TG_small = nx.DiGraph()
for _, row in top_transition_edges.iterrows():
    TG_small.add_edge(row["from_class"], row["to_class"], weight=int(row["count"]))

plt.figure(figsize=(10, 7), dpi=180)
pos = nx.spring_layout(TG_small, seed=42, k=1.2)

node_sizes = [1500 + 150 * TG_small.degree(n) for n in TG_small.nodes()]
edge_widths = [0.5 + 0.15 * TG_small[u][v]["weight"] for u, v in TG_small.edges()]

nx.draw_networkx_nodes(
    TG_small, pos,
    node_size=node_sizes,
    node_color="lightyellow",
    edgecolors="black",
    linewidths=0.7
)
nx.draw_networkx_edges(
    TG_small, pos,
    width=edge_widths,
    alpha=0.5,
    arrows=True,
    arrowsize=14
)
nx.draw_networkx_labels(TG_small, pos, font_size=10)

edge_labels = {(u, v): TG_small[u][v]["weight"] for u, v in TG_small.edges()}
nx.draw_networkx_edge_labels(TG_small, pos, edge_labels=edge_labels, font_size=8)

plt.title("Class transition graph across forensic alert sequence")
plt.axis("off")
plt.tight_layout()
timeline_plot4_png = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_class_transition_graph.png")
timeline_plot4_pdf = os.path.join(OUT_DIR, f"{TIMELINE_CASE_ID}_timeline_class_transition_graph.pdf")
plt.savefig(timeline_plot4_png, bbox_inches="tight")
plt.savefig(timeline_plot4_pdf, bbox_inches="tight")
plt.show()


# Earliest appearance of each class
first_seen_df = (
    timeline_df.groupby("prediction_name")["sequence_id"]
    .min()
    .reset_index()
    .rename(columns={"sequence_id": "first_seen_sequence"})
    .sort_values(by="first_seen_sequence", ascending=True)
    .reset_index(drop=True)
)

print("\n[First appearance of each class in forensic sequence]")
print(first_seen_df)


# Peak window per class
peak_rows = []
for cname in window_pivot.columns:
    peak_window = int(window_pivot[cname].idxmax())
    peak_count = int(window_pivot[cname].max())
    peak_rows.append({
        "class_name": cname,
        "peak_window": peak_window,
        "peak_count": peak_count
    })

peak_window_df = pd.DataFrame(peak_rows).sort_values(
    by=["peak_count", "peak_window"],
    ascending=[False, True]
).reset_index(drop=True)

print("\n[Peak sequence window per class]")
print(peak_window_df)


# Save outputs
window_pivot.to_csv(os.path.join(OUT_DIR, "timeline_window_class_counts.csv"))
transition_df.to_csv(os.path.join(OUT_DIR, "timeline_class_transitions.csv"), index=False)
first_seen_df.to_csv(os.path.join(OUT_DIR, "timeline_first_seen_classes.csv"), index=False)
peak_window_df.to_csv(os.path.join(OUT_DIR, "timeline_peak_windows.csv"), index=False)
timeline_df.to_csv(os.path.join(OUT_DIR, "timeline_evidence_sequence.csv"), index=False)

step8_summary = {
    "case_id": TIMELINE_CASE_ID,
    "timeline_evidence_log": TIMELINE_EVIDENCE_LOG,
    "logged_alerts": int(len(timeline_df)),
    "timeline_windows": int(N_WINDOWS),
    "classes_present": sorted(timeline_df["prediction_name"].dropna().unique().tolist()),
    "timeline_type": "sequence_based_forensic_timeline",
    "top_transition_edges": top_transition_edges.head(10).to_dict(orient="records"),
    "plot_files": [
        timeline_plot1_png, timeline_plot1_pdf,
        timeline_plot2_png, timeline_plot2_pdf,
        timeline_plot3_png, timeline_plot3_pdf,
        timeline_plot4_png, timeline_plot4_pdf
    ]
}

with open(os.path.join(OUT_DIR, "step8_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step8_summary, f, indent=2)

t_step8 = time.perf_counter() - t0

print("\n[Saved outputs]")
print("Directory:", OUT_DIR)
print("Plot files:")
for p in step8_summary["plot_files"]:
    print(p)

print("\n[Step 8 summary]")
print(json.dumps(step8_summary, indent=2))

print(f"\nSTEP 8 completed successfully in {t_step8:.4f} seconds.")

In [ ]:
# STEP 9: FEATURE-GROUP ABLATION STUDY (FINAL MODEL CONSISTENT)
# - uses LightGBM (selected final model)
# - tests dependence on major feature groups


import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

print("=" * 80)
print("STEP 9: FEATURE-GROUP ABLATION STUDY")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    "X_train_raw", "X_val_raw", "X_test_raw",
    "y_train", "y_val", "y_test",
    "inv_label_mapping"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError(f"LightGBM not available: {e}")


# Frequency encoder
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.freq_maps_ = {}

        n = len(X)
        for c in self.columns_:
            s = X[c].astype(str)
            vc = s.value_counts(dropna=False)
            self.freq_maps_[c] = (vc / max(n, 1)).to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = np.zeros((len(X), len(self.columns_)), dtype=float)

        for j, c in enumerate(self.columns_):
            fmap = self.freq_maps_.get(c, {})
            out[:, j] = X[c].astype(str).map(fmap).fillna(0.0).astype(float).to_numpy()

        return out


# Evaluation helper
def evaluate_multiclass(model, X, y):
    y_pred = model.predict(X)

    res = {
        "acc": accuracy_score(y, y_pred),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y, y_pred, average="weighted", zero_division=0),
        "prec_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "rec_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "roc_auc_ovr_macro": np.nan
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X)
            res["roc_auc_ovr_macro"] = roc_auc_score(
                y, y_proba, multi_class="ovr", average="macro"
            )
        except Exception:
            pass

    return res

# Build LightGBM pipeline for arbitrary feature subset
def build_lgbm_pipeline_for_features(X_train_subset):
    numeric_cols_local = X_train_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols_local = X_train_subset.select_dtypes(exclude=[np.number]).columns.tolist()

    log_transform_candidates = [
        "duration", "src_bytes", "dst_bytes", "missed_bytes",
        "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes",
        "http_request_body_len", "http_response_body_len"
    ]
    log_num_cols = [c for c in log_transform_candidates if c in numeric_cols_local]
    plain_num_cols = [c for c in numeric_cols_local if c not in log_num_cols]

    cat_cardinality = pd.DataFrame({
        "column": categorical_cols_local,
        "n_unique": [X_train_subset[c].nunique(dropna=False) for c in categorical_cols_local]
    }).sort_values(by="n_unique", ascending=False)

    HIGH_CARD_THRESHOLD = 50
    high_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] > HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    low_mid_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] <= HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    log_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, validate=False))
    ])

    plain_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    ohe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])

    freq_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("freq", FrequencyEncoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("log_num", log_num_pipe, log_num_cols),
            ("plain_num", plain_num_pipe, plain_num_cols),
            ("lowmid_cat", ohe_pipe, low_mid_card_cat_cols_local),
            ("high_cat", freq_pipe, high_card_cat_cols_local),
        ],
        remainder="drop"
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LGBMClassifier(
            objective="multiclass",
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
            verbosity=-1
        ))
    ])

    feature_meta = {
        "numeric_cols": numeric_cols_local,
        "categorical_cols": categorical_cols_local,
        "log_num_cols": log_num_cols,
        "plain_num_cols": plain_num_cols,
        "high_card_cat_cols": high_card_cat_cols_local,
        "low_mid_card_cat_cols": low_mid_card_cat_cols_local
    }

    return model, feature_meta


# Define ablation scenarios
ABLATION_SCENARIOS = {
    "full_features": [],
    "remove_transport_service_context": ["src_port", "dst_port", "service", "proto"],
    "remove_volume_features": ["src_ip_bytes", "src_bytes", "dst_ip_bytes", "dst_pkts", "src_pkts", "dst_bytes"],
    "remove_connection_state": ["conn_state"],
    "remove_query_features": ["dns_query", "dns_qtype", "dns_qclass", "dns_rcode", "dns_AA", "dns_RD", "dns_RA", "dns_rejected"],
    "remove_core_top_features": ["conn_state", "src_ip_bytes", "src_port", "dst_port", "duration", "service"]
}

print("\n[Ablation scenarios]")
for k, v in ABLATION_SCENARIOS.items():
    print(f"{k}: {v}")


# Train/evaluate each scenario
ablation_rows = []

for scenario_name, remove_cols in ABLATION_SCENARIOS.items():
    print("\n" + "-" * 80)
    print("Running ablation scenario:", scenario_name)
    print("Removed columns:", remove_cols if remove_cols else "None")

    X_train_ab = X_train_raw.drop(columns=[c for c in remove_cols if c in X_train_raw.columns], errors="ignore").copy()
    X_val_ab   = X_val_raw.drop(columns=[c for c in remove_cols if c in X_val_raw.columns], errors="ignore").copy()
    X_test_ab  = X_test_raw.drop(columns=[c for c in remove_cols if c in X_test_raw.columns], errors="ignore").copy()

    model_ab, feature_meta = build_lgbm_pipeline_for_features(X_train_ab)

    print("Remaining features:", X_train_ab.shape[1])
    print("Numeric features  :", len(feature_meta["numeric_cols"]))
    print("Categorical feats :", len(feature_meta["categorical_cols"]))

    # fit on train
    t_fit0 = time.perf_counter()
    model_ab.fit(X_train_ab, y_train)
    fit_s = time.perf_counter() - t_fit0

    # validation metrics
    val_res = evaluate_multiclass(model_ab, X_val_ab, y_val)

    # refit on train+val
    X_trainval_ab = pd.concat([X_train_ab, X_val_ab], axis=0).reset_index(drop=True)
    y_trainval_ab = pd.concat([pd.Series(y_train), pd.Series(y_val)], axis=0).reset_index(drop=True)

    t_refit0 = time.perf_counter()
    model_ab.fit(X_trainval_ab, y_trainval_ab)
    refit_s = time.perf_counter() - t_refit0

    # test metrics
    t_test0 = time.perf_counter()
    test_res = evaluate_multiclass(model_ab, X_test_ab, y_test)
    test_eval_s = time.perf_counter() - t_test0

    ablation_rows.append({
        "scenario": scenario_name,
        "removed_columns": ", ".join(remove_cols) if remove_cols else "None",
        "remaining_feature_count": X_train_ab.shape[1],

        "val_acc": val_res["acc"],
        "val_f1_macro": val_res["f1_macro"],
        "val_f1_weighted": val_res["f1_weighted"],
        "val_prec_macro": val_res["prec_macro"],
        "val_rec_macro": val_res["rec_macro"],
        "val_roc_auc_ovr_macro": val_res["roc_auc_ovr_macro"],

        "test_acc": test_res["acc"],
        "test_f1_macro": test_res["f1_macro"],
        "test_f1_weighted": test_res["f1_weighted"],
        "test_prec_macro": test_res["prec_macro"],
        "test_rec_macro": test_res["rec_macro"],
        "test_roc_auc_ovr_macro": test_res["roc_auc_ovr_macro"],

        "fit_s": fit_s,
        "refit_s": refit_s,
        "test_eval_s": test_eval_s
    })

    print(f"Validation macro-F1 : {val_res['f1_macro']:.6f}")
    print(f"Test macro-F1       : {test_res['f1_macro']:.6f}")
    print(f"Test accuracy       : {test_res['acc']:.6f}")


# Build results table
ablation_df = pd.DataFrame(ablation_rows)

full_row = ablation_df[ablation_df["scenario"] == "full_features"].iloc[0]

for col in ["test_acc", "test_f1_macro", "test_f1_weighted", "test_roc_auc_ovr_macro"]:
    ablation_df[f"delta_{col}"] = ablation_df[col] - full_row[col]

ablation_df = ablation_df.sort_values(by="test_f1_macro", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("[ABLATION RESULTS TABLE]")
print("=" * 80)
print(ablation_df)


# Compact table
ablation_compact_df = ablation_df[[
    "scenario",
    "removed_columns",
    "remaining_feature_count",
    "test_acc",
    "test_f1_macro",
    "test_f1_weighted",
    "test_roc_auc_ovr_macro",
    "delta_test_acc",
    "delta_test_f1_macro"
]].copy()

print("\n[Compact ablation summary]")
print(ablation_compact_df)


# Plot macro-F1 drop
plot_df = ablation_df.copy()
plot_df["f1_drop_vs_full"] = full_row["test_f1_macro"] - plot_df["test_f1_macro"]

plt.figure(figsize=(10, 6), dpi=180)
plt.barh(plot_df["scenario"][::-1], plot_df["f1_drop_vs_full"][::-1])
plt.title("Macro-F1 drop under feature-group ablation")
plt.xlabel("Drop in test macro-F1 relative to full model")
plt.ylabel("Ablation scenario")
plt.tight_layout()

OUT_DIR = "ablation_study_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

plot_png = os.path.join(OUT_DIR, "ablation_macro_f1_drop.png")
plot_pdf = os.path.join(OUT_DIR, "ablation_macro_f1_drop.pdf")
plt.savefig(plot_png, bbox_inches="tight")
plt.savefig(plot_pdf, bbox_inches="tight")
plt.show()


# Save results
ablation_df.to_csv(os.path.join(OUT_DIR, "ablation_results_full.csv"), index=False)
ablation_compact_df.to_csv(os.path.join(OUT_DIR, "ablation_results_compact.csv"), index=False)

step9_summary = {
    "ablation_model": "lightgbm",
    "full_model_test_macro_f1": float(full_row["test_f1_macro"]),
    "full_model_test_accuracy": float(full_row["test_acc"]),
    "scenarios": ablation_df[["scenario", "delta_test_acc", "delta_test_f1_macro"]].to_dict(orient="records"),
    "plot_files": [plot_png, plot_pdf]
}

with open(os.path.join(OUT_DIR, "step9_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step9_summary, f, indent=2)

t_step9 = time.perf_counter() - t0

print("\n[Saved outputs]")
print("Directory:", OUT_DIR)
print("Plot files:")
print(plot_png)
print(plot_pdf)

print("\n[Step 9 summary]")
print(json.dumps(step9_summary, indent=2))

print(f"\nSTEP 9 completed successfully in {t_step9:.4f} seconds.")

In [ ]:
# SYSTEM / EXPERIMENTAL ENVIRONMENT INFORMATION


import platform
import sys
import os
import psutil
import multiprocessing
import subprocess

print("="*70)
print("SYSTEM INFORMATION")
print("="*70)

print("Operating System:", platform.system(), platform.release())
print("OS Version:", platform.version())
print("Machine Architecture:", platform.machine())
print("Processor:", platform.processor())

print("\nCPU INFORMATION")
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("CPU Frequency:", psutil.cpu_freq())

print("\nMEMORY INFORMATION")
ram = psutil.virtual_memory()
print("Total RAM (GB):", round(ram.total / (1024**3), 2))
print("Available RAM (GB):", round(ram.available / (1024**3), 2))

print("\nPYTHON ENVIRONMENT")
print("Python Version:", sys.version)
print("Python Executable:", sys.executable)

print("\nKEY LIBRARY VERSIONS")

import numpy
import pandas
import sklearn
import matplotlib
import networkx

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("NetworkX:", networkx.__version__)

try:
    import shap
    print("SHAP:", shap.__version__)
except:
    print("SHAP: not installed")

try:
    import lime
    print("LIME:", lime.__version__)
except:
    print("LIME: not installed")

try:
    import tensorflow
    print("TensorFlow:", tensorflow.__version__)
except:
    print("TensorFlow: not installed")

print("\nGPU INFORMATION")

try:
    import torch
    if torch.cuda.is_available():
        print("CUDA Available:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available")
except:
    print("PyTorch not installed (GPU check skipped)")

print("\nENVIRONMENT PATH")
print(os.getcwd())

print("="*70)

In [ ]:
# SAVE ENVIRONMENT SNAPSHOT (REPRODUCIBILITY)


import json
from datetime import datetime

env_info = {
    "timestamp": str(datetime.utcnow()),

    "system": {
        "os": platform.system(),
        "os_release": platform.release(),
        "os_version": platform.version(),
        "architecture": platform.machine(),
        "processor": platform.processor()
    },

    "cpu": {
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_cores": psutil.cpu_count(logical=True),
        "frequency": str(psutil.cpu_freq())
    },

    "memory": {
        "total_gb": round(ram.total / (1024**3), 2),
        "available_gb": round(ram.available / (1024**3), 2)
    },

    "python": {
        "version": sys.version,
        "executable": sys.executable
    },

    "libraries": {
        "numpy": numpy.__version__,
        "pandas": pandas.__version__,
        "sklearn": sklearn.__version__,
        "matplotlib": matplotlib.__version__,
        "networkx": networkx.__version__,
        "shap": getattr(shap, "__version__", "not_installed") if 'shap' in globals() else "not_installed"
    },

    "gpu": "available" if ("torch" in sys.modules and torch.cuda.is_available()) else "not_available",

    "working_directory": os.getcwd()
}

OUT_DIR = "experiment_metadata"
os.makedirs(OUT_DIR, exist_ok=True)

env_path = os.path.join(OUT_DIR, "system_environment.json")

with open(env_path, "w") as f:
    json.dump(env_info, f, indent=2)

print("\n[Saved environment snapshot]")
print(env_path)

In [ ]:
# STEP 10: REALISTIC BINARY BENIGN-MAJORITY EVALUATION
# - 95:5 and 99:1 benign:attack test scenarios
# - LightGBM binary detector
# - PR-AUC, ROC-AUC, FPR at fixed TPR, threshold tuning

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

print("=" * 80)
print("STEP 10: REALISTIC BINARY BENIGN-MAJORITY EVALUATION")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = ["df_work"]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found. Run previous steps first."

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError(f"LightGBM not available: {e}")

# Frequency encoder
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.freq_maps_ = {}

        n = len(X)
        for c in self.columns_:
            s = X[c].astype(str)
            vc = s.value_counts(dropna=False)
            self.freq_maps_[c] = (vc / max(n, 1)).to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = np.zeros((len(X), len(self.columns_)), dtype=float)

        for j, c in enumerate(self.columns_):
            fmap = self.freq_maps_.get(c, {})
            out[:, j] = X[c].astype(str).map(fmap).fillna(0.0).astype(float).to_numpy()

        return out


# Build binary dataset from deduplicated data
df_bin = df_work.copy().reset_index(drop=True)
df_bin["binary_target"] = pd.to_numeric(df_bin["label"], errors="coerce").fillna(-1).astype(int)

print("\n[Binary dataset]")
print("Shape:", df_bin.shape)
print("Binary class distribution:")
print(df_bin["binary_target"].value_counts())
print((100 * df_bin["binary_target"].value_counts(normalize=True)).round(4))


# Preserve context columns

CONTEXT_COLS_BIN = [c for c in [
    "src_ip", "dst_ip", "src_port", "dst_port", "proto", "service", "conn_state"
] if c in df_bin.columns]


# Feature exclusions

DROP_FROM_FEATURES_BIN = [c for c in [
    "type",
    "label",
    "binary_target",
    "src_ip",
    "dst_ip"
] if c in df_bin.columns]

X_bin_raw = df_bin.drop(columns=DROP_FROM_FEATURES_BIN, errors="ignore").copy()
y_bin = df_bin["binary_target"].copy()

obj_cols_bin = X_bin_raw.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols_bin:
    X_bin_raw[c] = X_bin_raw[c].astype(str).str.strip()
    X_bin_raw[c] = X_bin_raw[c].replace({"-": "MISSING_TOKEN", "": "MISSING_TOKEN", "nan": "MISSING_TOKEN"})


# Split into benign and attack pools
# Use random row split here for deployment-style class ratio simulation
# The grouped binary split from Step 2C remains your leakage-resistant branch

BENIGN_CLASS = 0
ATTACK_CLASS = 1

benign_idx = np.where(y_bin.values == BENIGN_CLASS)[0]
attack_idx = np.where(y_bin.values == ATTACK_CLASS)[0]

rng = np.random.default_rng(42)
rng.shuffle(benign_idx)
rng.shuffle(attack_idx)

print("\n[Pool sizes]")
print("Benign pool:", len(benign_idx))
print("Attack pool:", len(attack_idx))


# Build a reasonably balanced training set
# and separate imbalanced test scenarios

N_TRAIN_PER_CLASS = min(20000, len(benign_idx) // 2, len(attack_idx) // 4)
N_VAL_PER_CLASS = min(5000, len(benign_idx) // 6, len(attack_idx) // 10)

assert N_TRAIN_PER_CLASS > 1000, "Not enough data for stable balanced training."
assert N_VAL_PER_CLASS > 500, "Not enough data for stable validation."

train_benign_idx = benign_idx[:N_TRAIN_PER_CLASS]
val_benign_idx   = benign_idx[N_TRAIN_PER_CLASS:N_TRAIN_PER_CLASS + N_VAL_PER_CLASS]

train_attack_idx = attack_idx[:N_TRAIN_PER_CLASS]
val_attack_idx   = attack_idx[N_TRAIN_PER_CLASS:N_TRAIN_PER_CLASS + N_VAL_PER_CLASS]

used_idx = set(train_benign_idx) | set(val_benign_idx) | set(train_attack_idx) | set(val_attack_idx)

remaining_benign_idx = np.array([i for i in benign_idx if i not in used_idx], dtype=int)
remaining_attack_idx = np.array([i for i in attack_idx if i not in used_idx], dtype=int)

print("\n[Balanced training/validation design]")
print("Train per class:", N_TRAIN_PER_CLASS)
print("Val per class  :", N_VAL_PER_CLASS)
print("Remaining benign pool:", len(remaining_benign_idx))
print("Remaining attack pool:", len(remaining_attack_idx))


# Construct train and val
train_idx = np.concatenate([train_benign_idx, train_attack_idx])
val_idx   = np.concatenate([val_benign_idx, val_attack_idx])

rng.shuffle(train_idx)
rng.shuffle(val_idx)

X_train_bin = X_bin_raw.iloc[train_idx].reset_index(drop=True)
y_train_bin = y_bin.iloc[train_idx].reset_index(drop=True)

X_val_bin   = X_bin_raw.iloc[val_idx].reset_index(drop=True)
y_val_bin   = y_bin.iloc[val_idx].reset_index(drop=True)

ctx_train_bin_realistic = df_bin.iloc[train_idx][CONTEXT_COLS_BIN].reset_index(drop=True)
ctx_val_bin_realistic   = df_bin.iloc[val_idx][CONTEXT_COLS_BIN].reset_index(drop=True)

print("\n[Balanced train/val shapes]")
print("X_train_bin:", X_train_bin.shape)
print("X_val_bin  :", X_val_bin.shape)


# Construct imbalanced test sets
def make_imbalanced_test_set(benign_ratio=0.95, total_size=20000, seed=42):
    rng_local = np.random.default_rng(seed)

    n_benign = int(round(total_size * benign_ratio))
    n_attack = total_size - n_benign

    n_benign = min(n_benign, len(remaining_benign_idx))
    n_attack = min(n_attack, len(remaining_attack_idx))

    idx_b = remaining_benign_idx[:n_benign]
    idx_a = remaining_attack_idx[:n_attack]

    idx = np.concatenate([idx_b, idx_a])
    rng_local.shuffle(idx)

    X = X_bin_raw.iloc[idx].reset_index(drop=True)
    y = y_bin.iloc[idx].reset_index(drop=True)
    ctx = df_bin.iloc[idx][CONTEXT_COLS_BIN].reset_index(drop=True)

    return X, y, ctx, idx

X_test_95, y_test_95, ctx_test_95, idx_test_95 = make_imbalanced_test_set(benign_ratio=0.95, total_size=20000, seed=42)
X_test_99, y_test_99, ctx_test_99, idx_test_99 = make_imbalanced_test_set(benign_ratio=0.99, total_size=20000, seed=43)

print("\n[Imbalanced test scenario shapes]")
print("95:5 test shape:", X_test_95.shape, "| benign ratio:", round((y_test_95 == 0).mean(), 4))
print("99:1 test shape:", X_test_99.shape, "| benign ratio:", round((y_test_99 == 0).mean(), 4))


# Build binary LightGBM pipeline
def build_binary_lgbm_pipeline(X_train_subset):
    numeric_cols_local = X_train_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols_local = X_train_subset.select_dtypes(exclude=[np.number]).columns.tolist()

    log_transform_candidates = [
        "duration", "src_bytes", "dst_bytes", "missed_bytes",
        "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes",
        "http_request_body_len", "http_response_body_len"
    ]
    log_num_cols = [c for c in log_transform_candidates if c in numeric_cols_local]
    plain_num_cols = [c for c in numeric_cols_local if c not in log_num_cols]

    cat_cardinality = pd.DataFrame({
        "column": categorical_cols_local,
        "n_unique": [X_train_subset[c].nunique(dropna=False) for c in categorical_cols_local]
    }).sort_values(by="n_unique", ascending=False)

    HIGH_CARD_THRESHOLD = 50
    high_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] > HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    low_mid_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] <= HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    log_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, validate=False))
    ])

    plain_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    ohe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])

    freq_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("freq", FrequencyEncoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("log_num", log_num_pipe, log_num_cols),
            ("plain_num", plain_num_pipe, plain_num_cols),
            ("lowmid_cat", ohe_pipe, low_mid_card_cat_cols_local),
            ("high_cat", freq_pipe, high_card_cat_cols_local),
        ],
        remainder="drop"
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LGBMClassifier(
            objective="binary",
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
            verbosity=-1
        ))
    ])

    return model

binary_model = build_binary_lgbm_pipeline(X_train_bin)


# Fit binary model
t_fit0 = time.perf_counter()
binary_model.fit(X_train_bin, y_train_bin)
fit_s = time.perf_counter() - t_fit0

print("\n[Binary model fitted]")
print(f"Fit time (s): {fit_s:.4f}")


# Binary evaluation helpers

def compute_binary_metrics(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / max(fp + tn, 1)
    tpr = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)

    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall_tpr": float(rec),
        "specificity": float(specificity),
        "fpr": float(fpr),
        "tpr": float(tpr),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

def find_threshold_for_target_tpr(y_true, y_score, target_tpr):
    fpr_arr, tpr_arr, thr_arr = roc_curve(y_true, y_score)
    valid = np.where(tpr_arr >= target_tpr)[0]
    if len(valid) == 0:
        return None

    idx = valid[np.argmin(fpr_arr[valid])]
    return {
        "target_tpr": float(target_tpr),
        "threshold": float(thr_arr[idx]),
        "achieved_tpr": float(tpr_arr[idx]),
        "achieved_fpr": float(fpr_arr[idx])
    }

def find_best_threshold_under_fpr(y_true, y_score, max_fpr=0.01):
    thresholds = np.linspace(0, 1, 1001)
    best = None

    for th in thresholds:
        m = compute_binary_metrics(y_true, y_score, threshold=th)
        if m["fpr"] <= max_fpr:
            if best is None or m["f1"] > best["f1"]:
                best = m

    return best

def evaluate_scenario(model, X, y, scenario_name):
    t_pred0 = time.perf_counter()
    y_score = model.predict_proba(X)[:, 1]
    pred_s = time.perf_counter() - t_pred0

    default_metrics = compute_binary_metrics(y, y_score, threshold=0.5)
    default_metrics["scenario"] = scenario_name
    default_metrics["pred_time_s"] = float(pred_s)
    default_metrics["pred_ms_per_sample"] = float(1000 * pred_s / len(X))

    tpr95 = find_threshold_for_target_tpr(y, y_score, target_tpr=0.95)
    tpr99 = find_threshold_for_target_tpr(y, y_score, target_tpr=0.99)
    best_under_fpr_1pct = find_best_threshold_under_fpr(y, y_score, max_fpr=0.01)

    precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y, y_score)
    fpr_arr, tpr_arr, roc_thresholds = roc_curve(y, y_score)

    curves = {
        "precision": precision_arr,
        "recall": recall_arr,
        "pr_thresholds": pr_thresholds,
        "fpr": fpr_arr,
        "tpr": tpr_arr,
        "roc_thresholds": roc_thresholds
    }

    return default_metrics, tpr95, tpr99, best_under_fpr_1pct, curves, y_score


# Evaluate 95:5 and 99:1 scenarios
default_95, tpr95_95, tpr99_95, best_fpr1_95, curves_95, y_score_95 = evaluate_scenario(
    binary_model, X_test_95, y_test_95, "95_5_benign_attack"
)
default_99, tpr95_99, tpr99_99, best_fpr1_99, curves_99, y_score_99 = evaluate_scenario(
    binary_model, X_test_99, y_test_99, "99_1_benign_attack"
)

print("\n[Default threshold metrics @ 0.5]")
print(pd.DataFrame([default_95, default_99]))

print("\n[Thresholds for target TPR]")
print("95:5 @ TPR>=0.95:", tpr95_95)
print("95:5 @ TPR>=0.99:", tpr99_95)
print("99:1 @ TPR>=0.95:", tpr95_99)
print("99:1 @ TPR>=0.99:", tpr99_99)

print("\n[Best threshold under FPR <= 1%]")
print("95:5:", best_fpr1_95)
print("99:1:", best_fpr1_99)


# Plot ROC curves
OUT_DIR = "binary_benign_majority_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

plt.figure(figsize=(8, 6), dpi=180)
plt.plot(curves_95["fpr"], curves_95["tpr"], label=f"95:5 (ROC-AUC={default_95['roc_auc']:.4f})")
plt.plot(curves_99["fpr"], curves_99["tpr"], label=f"99:1 (ROC-AUC={default_99['roc_auc']:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves under benign-majority scenarios")
plt.legend()
plt.tight_layout()
roc_png = os.path.join(OUT_DIR, "binary_benign_majority_roc.png")
roc_pdf = os.path.join(OUT_DIR, "binary_benign_majority_roc.pdf")
plt.savefig(roc_png, bbox_inches="tight")
plt.savefig(roc_pdf, bbox_inches="tight")
plt.show()


# Plot PR curves
plt.figure(figsize=(8, 6), dpi=180)
plt.plot(curves_95["recall"], curves_95["precision"], label=f"95:5 (PR-AUC={default_95['pr_auc']:.4f})")
plt.plot(curves_99["recall"], curves_99["precision"], label=f"99:1 (PR-AUC={default_99['pr_auc']:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves under benign-majority scenarios")
plt.legend()
plt.tight_layout()
pr_png = os.path.join(OUT_DIR, "binary_benign_majority_pr.png")
pr_pdf = os.path.join(OUT_DIR, "binary_benign_majority_pr.pdf")
plt.savefig(pr_png, bbox_inches="tight")
plt.savefig(pr_pdf, bbox_inches="tight")
plt.show()


# Build summary tables
default_df = pd.DataFrame([default_95, default_99])

threshold_rows = []
for scenario, res in [
    ("95:5_tpr95", tpr95_95),
    ("95:5_tpr99", tpr99_95),
    ("99:1_tpr95", tpr95_99),
    ("99:1_tpr99", tpr99_99),
]:
    if res is not None:
        row = {"scenario": scenario}
        row.update(res)
        threshold_rows.append(row)

threshold_targets_df = pd.DataFrame(threshold_rows)

fpr1_rows = []
for scenario, res in [
    ("95:5_fpr<=1%", best_fpr1_95),
    ("99:1_fpr<=1%", best_fpr1_99),
]:
    if res is not None:
        row = {"scenario": scenario}
        row.update(res)
        fpr1_rows.append(row)

fpr1_df = pd.DataFrame(fpr1_rows)

print("\n[Default metrics table]")
print(default_df)

print("\n[Target-TPR threshold table]")
print(threshold_targets_df)

print("\n[FPR-constrained threshold table]")
print(fpr1_df)


# Save outputs
default_df.to_csv(os.path.join(OUT_DIR, "default_threshold_metrics.csv"), index=False)
threshold_targets_df.to_csv(os.path.join(OUT_DIR, "target_tpr_thresholds.csv"), index=False)
fpr1_df.to_csv(os.path.join(OUT_DIR, "fpr_constrained_thresholds.csv"), index=False)

step10_summary = {
    "binary_model": "lightgbm",
    "fit_time_seconds": float(fit_s),
    "train_shape": list(X_train_bin.shape),
    "val_shape": list(X_val_bin.shape),
    "test_95_shape": list(X_test_95.shape),
    "test_99_shape": list(X_test_99.shape),
    "default_metrics": default_df.to_dict(orient="records"),
    "target_tpr_thresholds": threshold_targets_df.to_dict(orient="records"),
    "fpr_constrained_thresholds": fpr1_df.to_dict(orient="records"),
    "plot_files": [roc_png, roc_pdf, pr_png, pr_pdf]
}

with open(os.path.join(OUT_DIR, "step10_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step10_summary, f, indent=2)

t_step10 = time.perf_counter() - t0

print("\n[Saved outputs]")
print("Directory:", OUT_DIR)
print("Plot files:")
for p in step10_summary["plot_files"]:
    print(p)

print("\n[Step 10 summary]")
print(json.dumps(step10_summary, indent=2))

print(f"\nSTEP 10 completed successfully in {t_step10:.4f} seconds.")

In [ ]:
# STEP 11: MULTI-SEED STABILITY ANALYSIS
# - final-model robustness across repeated runs
# - multiclass stability
# - binary 95:5 and 99:1 stability
# - reports mean ± std

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 80)
print("STEP 11: MULTI-SEED STABILITY ANALYSIS")
print("=" * 80)

t0 = time.perf_counter()


# Safety checks
required_vars = [
    # multiclass
    "X_train_raw", "X_val_raw", "X_test_raw",
    "y_train", "y_val", "y_test",
    # binary
    "X_train_bin", "X_val_bin", "X_test_95", "X_test_99",
    "y_train_bin", "y_val_bin", "y_test_95", "y_test_99"
]
for v in required_vars:
    assert v in globals(), f"Required variable '{v}' not found."

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError(f"LightGBM not available: {e}")


# Frequency encoder
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.freq_maps_ = {}

        n = len(X)
        for c in self.columns_:
            s = X[c].astype(str)
            vc = s.value_counts(dropna=False)
            self.freq_maps_[c] = (vc / max(n, 1)).to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = np.zeros((len(X), len(self.columns_)), dtype=float)

        for j, c in enumerate(self.columns_):
            fmap = self.freq_maps_.get(c, {})
            out[:, j] = X[c].astype(str).map(fmap).fillna(0.0).astype(float).to_numpy()

        return out


# Generic pipeline builder
def build_lgbm_pipeline(X_train_subset, objective="multiclass", seed=42):
    numeric_cols_local = X_train_subset.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols_local = X_train_subset.select_dtypes(exclude=[np.number]).columns.tolist()

    log_transform_candidates = [
        "duration", "src_bytes", "dst_bytes", "missed_bytes",
        "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes",
        "http_request_body_len", "http_response_body_len"
    ]
    log_num_cols = [c for c in log_transform_candidates if c in numeric_cols_local]
    plain_num_cols = [c for c in numeric_cols_local if c not in log_num_cols]

    cat_cardinality = pd.DataFrame({
        "column": categorical_cols_local,
        "n_unique": [X_train_subset[c].nunique(dropna=False) for c in categorical_cols_local]
    }).sort_values(by="n_unique", ascending=False)

    HIGH_CARD_THRESHOLD = 50
    high_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] > HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    low_mid_card_cat_cols_local = cat_cardinality.loc[
        cat_cardinality["n_unique"] <= HIGH_CARD_THRESHOLD, "column"
    ].tolist()

    log_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, validate=False))
    ])

    plain_num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    ohe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])

    freq_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("freq", FrequencyEncoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("log_num", log_num_pipe, log_num_cols),
            ("plain_num", plain_num_pipe, plain_num_cols),
            ("lowmid_cat", ohe_pipe, low_mid_card_cat_cols_local),
            ("high_cat", freq_pipe, high_card_cat_cols_local),
        ],
        remainder="drop"
    )

    if objective == "multiclass":
        clf = LGBMClassifier(
            objective="multiclass",
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed,
            n_jobs=-1,
            class_weight="balanced",
            verbosity=-1
        )
    elif objective == "binary":
        clf = LGBMClassifier(
            objective="binary",
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed,
            n_jobs=-1,
            class_weight="balanced",
            verbosity=-1
        )
    else:
        raise ValueError(f"Unsupported objective: {objective}")

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("clf", clf)
    ])

    return model


# Evaluation helpers
def eval_multiclass(model, X, y):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    return {
        "acc": accuracy_score(y, y_pred),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y, y_pred, average="weighted", zero_division=0),
        "prec_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "rec_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "roc_auc_ovr_macro": roc_auc_score(y, y_proba, multi_class="ovr", average="macro")
    }

def eval_binary(model, X, y, threshold=0.5):
    y_score = model.predict_proba(X)[:, 1]
    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()

    return {
        "acc": accuracy_score(y, y_pred),
        "f1": f1_score(y, y_pred, zero_division=0),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_tpr": recall_score(y, y_pred, zero_division=0),
        "fpr": fp / max(fp + tn, 1),
        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score)
    }


# Seed list
SEEDS = [7, 21, 42, 84, 168]

print("\n[Seeds]")
print(SEEDS)


# Multi-seed multiclass stability
multiclass_rows = []

X_trainval_mc = pd.concat([X_train_raw, X_val_raw], axis=0).reset_index(drop=True)
y_trainval_mc = pd.concat([pd.Series(y_train), pd.Series(y_val)], axis=0).reset_index(drop=True)

for seed in SEEDS:
    print("\n" + "-" * 80)
    print(f"[Multiclass] Running seed = {seed}")

    model_mc = build_lgbm_pipeline(X_trainval_mc, objective="multiclass", seed=seed)

    t_fit0 = time.perf_counter()
    model_mc.fit(X_trainval_mc, y_trainval_mc)
    fit_s = time.perf_counter() - t_fit0

    res = eval_multiclass(model_mc, X_test_raw, y_test)
    res["seed"] = seed
    res["fit_s"] = fit_s
    multiclass_rows.append(res)

    print(f"Test macro-F1 : {res['f1_macro']:.6f}")
    print(f"Test accuracy : {res['acc']:.6f}")
    print(f"ROC-AUC macro : {res['roc_auc_ovr_macro']:.6f}")

multiclass_seed_df = pd.DataFrame(multiclass_rows)


# Multi-seed binary stability
# Use the same balanced train+val binary pool, then evaluate on 95:5 and 99:1
binary_rows = []

X_trainval_bin = pd.concat([X_train_bin, X_val_bin], axis=0).reset_index(drop=True)
y_trainval_bin = pd.concat([pd.Series(y_train_bin), pd.Series(y_val_bin)], axis=0).reset_index(drop=True)

for seed in SEEDS:
    print("\n" + "-" * 80)
    print(f"[Binary] Running seed = {seed}")

    model_bin = build_lgbm_pipeline(X_trainval_bin, objective="binary", seed=seed)

    t_fit0 = time.perf_counter()
    model_bin.fit(X_trainval_bin, y_trainval_bin)
    fit_s = time.perf_counter() - t_fit0

    res_95 = eval_binary(model_bin, X_test_95, y_test_95, threshold=0.5)
    res_95["seed"] = seed
    res_95["scenario"] = "95_5"
    res_95["fit_s"] = fit_s
    binary_rows.append(res_95)

    res_99 = eval_binary(model_bin, X_test_99, y_test_99, threshold=0.5)
    res_99["seed"] = seed
    res_99["scenario"] = "99_1"
    res_99["fit_s"] = fit_s
    binary_rows.append(res_99)

    print(f"95:5 F1   : {res_95['f1']:.6f} | PR-AUC: {res_95['pr_auc']:.6f} | FPR: {res_95['fpr']:.6f}")
    print(f"99:1 F1   : {res_99['f1']:.6f} | PR-AUC: {res_99['pr_auc']:.6f} | FPR: {res_99['fpr']:.6f}")

binary_seed_df = pd.DataFrame(binary_rows)


# Summaries: mean ± std
def mean_std_table(df, metric_cols, group_cols=None):
    if group_cols is None:
        group_cols = []

    grouped = df.groupby(group_cols) if group_cols else [((), df)]

    rows = []
    for key, subdf in grouped:
        row = {}
        if group_cols:
            if len(group_cols) == 1:
                row[group_cols[0]] = key
            else:
                for k, v in zip(group_cols, key):
                    row[k] = v

        for col in metric_cols:
            row[f"{col}_mean"] = subdf[col].mean()
            row[f"{col}_std"] = subdf[col].std(ddof=1)

        rows.append(row)

    return pd.DataFrame(rows)

multiclass_metrics = ["acc", "f1_macro", "f1_weighted", "prec_macro", "rec_macro", "roc_auc_ovr_macro", "fit_s"]
binary_metrics = ["acc", "f1", "precision", "recall_tpr", "fpr", "roc_auc", "pr_auc", "fit_s"]

multiclass_summary_df = mean_std_table(multiclass_seed_df, multiclass_metrics)
binary_summary_df = mean_std_table(binary_seed_df, binary_metrics, group_cols=["scenario"])

print("\n" + "=" * 80)
print("[MULTICLASS SEED-WISE RESULTS]")
print("=" * 80)
print(multiclass_seed_df)

print("\n[MULTICLASS MEAN ± STD]")
print(multiclass_summary_df)

print("\n" + "=" * 80)
print("[BINARY SEED-WISE RESULTS]")
print("=" * 80)
print(binary_seed_df)

print("\n[BINARY MEAN ± STD]")
print(binary_summary_df)


# Compact tables
multiclass_compact_df = multiclass_summary_df[[
    "acc_mean", "acc_std",
    "f1_macro_mean", "f1_macro_std",
    "f1_weighted_mean", "f1_weighted_std",
    "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_std"
]].copy()

binary_compact_df = binary_summary_df[[
    "scenario",
    "acc_mean", "acc_std",
    "f1_mean", "f1_std",
    "precision_mean", "precision_std",
    "recall_tpr_mean", "recall_tpr_std",
    "fpr_mean", "fpr_std",
    "pr_auc_mean", "pr_auc_std"
]].copy()

print("\n[Compact multiclass stability summary]")
print(multiclass_compact_df)

print("\n[Compact binary stability summary]")
print(binary_compact_df)


# Plot: multiclass macro-F1 by seed
OUT_DIR = "multiseed_stability_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

plt.figure(figsize=(8, 5), dpi=180)
plt.plot(multiclass_seed_df["seed"], multiclass_seed_df["f1_macro"], marker="o")
plt.title("Multiclass macro-F1 across random seeds")
plt.xlabel("Seed")
plt.ylabel("Test macro-F1")
plt.tight_layout()
mc_plot_png = os.path.join(OUT_DIR, "multiclass_macro_f1_by_seed.png")
mc_plot_pdf = os.path.join(OUT_DIR, "multiclass_macro_f1_by_seed.pdf")
plt.savefig(mc_plot_png, bbox_inches="tight")
plt.savefig(mc_plot_pdf, bbox_inches="tight")
plt.show()


# Plot: binary PR-AUC by seed for both scenarios
plt.figure(figsize=(8, 5), dpi=180)
for scenario in ["95_5", "99_1"]:
    sub = binary_seed_df[binary_seed_df["scenario"] == scenario].sort_values("seed")
    plt.plot(sub["seed"], sub["pr_auc"], marker="o", label=scenario)

plt.title("Binary PR-AUC across random seeds")
plt.xlabel("Seed")
plt.ylabel("PR-AUC")
plt.legend()
plt.tight_layout()
bin_plot_png = os.path.join(OUT_DIR, "binary_pr_auc_by_seed.png")
bin_plot_pdf = os.path.join(OUT_DIR, "binary_pr_auc_by_seed.pdf")
plt.savefig(bin_plot_png, bbox_inches="tight")
plt.savefig(bin_plot_pdf, bbox_inches="tight")
plt.show()


# Save outputs
multiclass_seed_df.to_csv(os.path.join(OUT_DIR, "multiclass_seedwise_results.csv"), index=False)
multiclass_summary_df.to_csv(os.path.join(OUT_DIR, "multiclass_stability_summary.csv"), index=False)
multiclass_compact_df.to_csv(os.path.join(OUT_DIR, "multiclass_stability_compact.csv"), index=False)

binary_seed_df.to_csv(os.path.join(OUT_DIR, "binary_seedwise_results.csv"), index=False)
binary_summary_df.to_csv(os.path.join(OUT_DIR, "binary_stability_summary.csv"), index=False)
binary_compact_df.to_csv(os.path.join(OUT_DIR, "binary_stability_compact.csv"), index=False)

step11_summary = {
    "seeds": SEEDS,
    "multiclass_summary": multiclass_summary_df.to_dict(orient="records"),
    "binary_summary": binary_summary_df.to_dict(orient="records"),
    "plot_files": [mc_plot_png, mc_plot_pdf, bin_plot_png, bin_plot_pdf]
}

with open(os.path.join(OUT_DIR, "step11_summary.json"), "w", encoding="utf-8") as f:
    json.dump(step11_summary, f, indent=2)

t_step11 = time.perf_counter() - t0

print("\n[Saved outputs]")
print("Directory:", OUT_DIR)
print("Plot files:")
for p in step11_summary["plot_files"]:
    print(p)

print("\n[Step 11 summary]")
print(json.dumps(step11_summary, indent=2))

print(f"\nSTEP 11 completed successfully in {t_step11:.4f} seconds.")